# SMA-200 Pullback + Piotroski F-Score — Walk-Forward Backtest

Backtest of the strategy below on **daily** data, with a walk-forward (step-forward) optimisation of the free parameters and a set of checks for overfitting and lookahead bias. Only free data and free Python libraries are used.

### Strategy rules (defaults = your spec)
| Rule | Implementation |
|---|---|
| Universe | S&P 500 members **on each date** (point-in-time membership, not today's list) |
| Trend filter | `Close < SMA200` **and** `Close > SMA50` (optimised over **SMA20 / SMA50 / SMA100**) |
| Quality filter | Buy only if the Piotroski F-Score is **≥ 7** (optimised over **7 / 8**) |
| Ranking | Largest market cap first (`RANK_BY`; expected upside to the SMA200 and F-Score are compared in section 8) |
| Diversification | Max **1 stock per sector** (optimised over **1 / 2 / 3 / no limit**) |
| Portfolio | Max **5** positions (optimised over **3 / 5 / 8 / 10**), rest in cash |
| Position size | **Equal weight** (equity ÷ max positions), or **inverse volatility**, both optimised. Inverse volatility gives calm stocks more money and volatile stocks less, so each position carries a similar risk. |
| Take profit | **SMA200**: a resting sell limit at yesterday's SMA200 (fills intraday if the high reaches it, or at the open on a gap up). The SMA200 moves: if a stock keeps falling, its SMA200 falls too, and the exit can end up below the entry price. Those exits are logged as `SMA200 exit at a loss` |
| Quality exit | Exit at the next open when a new filing takes the F-Score **below 7** |
| Stop loss | **None** by default (`STOP_LOSS`). The ablation in section 8 still shows what a 10% stop would have done |
| Time limit | Sell at the open once a position has been held **252 trading days (1 year)**. Fixed in advance, **not optimised** |
| Market regime | **No new buys while SPY closes below its own 200-day SMA** (existing positions keep their exits). Fixed in advance, **not optimised** |

### How lookahead bias is avoided
1. **Signals use yesterday's close; trades happen at today's open.** SMAs, filters and ranks are all computed on data up to the previous close.
2. **Fundamentals are point-in-time.** F-Score inputs come from SEC EDGAR XBRL filings. Each value is dated by its **filing date** (not the fiscal period end), it is used only from the next trading day, and the **first-reported** number is kept, so later restatements are ignored.
3. **Market cap is point-in-time**: yesterday's close × the latest share count filed with the SEC, restated for splits.
4. **Index membership is point-in-time**, so stocks are not traded before they joined the S&P 500.
5. **Parameters are chosen walk-forward.** Each out-of-sample year is traded with the parameters that scored best on the preceding in-sample years only. The out-of-sample equity curve is the headline result.
6. **Automated audit.** The whole pipeline is rebuilt on data cut off at an earlier date; results up to that date must be identical (section 9).

### How overfitting is measured
A grid of 180 configurations (moving average × sector cap × portfolio size × sizing × minimum F-Score), walk-forward selection with a points scorecard, an ablation that re-runs the whole walk-forward with and without each risk control, Deflated Sharpe Ratio, Probability of Backtest Overfitting (CSCV), walk-forward efficiency, parameter-stability tables, block-bootstrap confidence intervals and a cost-sensitivity test.

### Data sources (all free)
* Prices, splits and dividends: **Yahoo Finance** via `yfinance`
* Fundamentals and share counts: **SEC EDGAR XBRL** `companyfacts` API (no key needed; the SEC asks for a contact e-mail in the User-Agent)
* Historical S&P 500 membership: [fja05680/sp500](https://github.com/fja05680/sp500) (GitHub); GICS sectors from its current-constituents file

**How to run:** set `SEC_USER_AGENT` in the config cell, then *Run All*. The first run downloads about 900 price histories and SEC files (roughly 15–30 minutes). Everything is cached in `data_cache/`, so later runs take a minute or two.

In [ ]:
# One-time setup (uncomment if packages are missing)
# %pip install yfinance pandas numpy scipy matplotlib requests

In [ ]:
%matplotlib inline
import os, json, gzip, time, itertools, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy import stats
from IPython.display import display, Markdown

try:
    import yfinance as yf
except ImportError:          # only needed when prices are not cached yet
    yf = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

## 1. Configuration

In [ ]:
# ---------------- Strategy (defaults = your rules) ----------------
MAX_POSITIONS   = 5          # max simultaneous holdings (optimised over PARAM_GRID["max_positions"])
F_MIN           = 7          # buy only if F-Score >= this (optimised over PARAM_GRID["f_min"])
F_EXIT          = 7          # sell when a new filing takes the F-Score below this
SMA_SLOW        = 200        # price must be BELOW this; it is also the take-profit level
SMA_FAST        = 50         # price must be ABOVE this (optimised over PARAM_GRID["sma_fast"])
MAX_PER_SECTOR  = 1          # max stocks held per sector; None = no limit (optimised over PARAM_GRID["max_per_sector"])
SIZING          = "equal"    # "equal": equity / max positions | "inverse_vol": scaled by 1 / recent volatility
VOL_LOOKBACK    = 60         # trading days of returns used to measure each stock's volatility
VOL_WEIGHT_BOUNDS = (0.5, 2.0)   # inverse-vol position size stays between 0.5x and 2x the equal-weight size
TP_MODE         = "limit"    # "limit": resting sell limit at yesterday's SMA200 | "close": sell next open after a close >= SMA200
FSCORE_BASIS    = "ttm"      # "ttm": trailing 12 months vs prior-year TTM | "quarter": latest quarter vs same quarter last year
EXCLUDE_SECTORS = ["Financials", "Real Estate"]   # F-Score is not meaningful for banks/insurers/REITs. Use [] to keep them.
RANK_BY         = "market_cap"  # candidate order: "market_cap" (largest first) | "upside" (furthest below SMA200) | "fscore"
ENTRY_TRIGGER   = "level"       # "level": F-Score >= minimum | "upgrade": latest filing lifted the F-Score from below 7

# ---------------- Risk controls (fixed in advance, deliberately NOT in the optimisation grid) ----------------
STOP_LOSS     = None              # e.g. 0.10 = sell when the price is 10% below the entry price; None = no stop
MARKET_FILTER = "no_new_entries"  # when BENCHMARK closes below its MARKET_FILTER_SMA: "no_new_entries" | "liquidate" | None
MARKET_FILTER_SMA = 200
MAX_HOLD_DAYS = 252               # sell at the open after this many trading days held (252 = 1 year); None = no limit

# ---------------- Walk-forward optimisation ----------------
PARAM_GRID   = {"sma_fast": [20, 50, 100],                 # lower trend bound
                "max_per_sector": [1, 2, 3, None],         # None = no sector limit
                "max_positions": [3, 5, 8, 10],
                "sizing": ["equal", "inverse_vol"],
                "f_min": [7, 8]}                           # 180 configurations (redundant combinations dropped)
WF_IS_YEARS  = 3             # in-sample window used to pick the parameters
WF_OOS_YEARS = 1             # out-of-sample window traded with them, then step forward
WF_ANCHORED  = False         # False: rolling IS window | True: expanding window from BACKTEST_START
WF_OBJECTIVE = "points"      # "points" (SCORECARD below) | "sharpe" | "sortino" | "calmar" | "cagr"
# Scorecard used each year to pick the configuration: over the in-sample window, the single best value of each
# metric earns its points (configurations tied on a metric all earn them). Most points (out of 6) moves forward;
# ties are broken by Sharpe. Configurations that never traded in the window are not eligible.
SCORECARD = [("Total return", 1.0, "max"),
             ("Max drawdown", 1.0, "max"),     # drawdowns are negative: the one closest to 0 wins
             ("Sharpe",       2.0, "max"),
             ("Calmar",       1.0, "max"),
             ("Beta",         0.5, "min"),     # vs BENCHMARK
             ("Sortino",      0.5, "max")]

# ---------------- Backtest ----------------
PRICE_START     = "2009-01-01"   # price history (SMA200 warm-up)
BACKTEST_START  = "2011-01-01"   # SEC XBRL data is complete for large caps from ~2010
BACKTEST_END    = None           # None = latest available data
INITIAL_CAPITAL = 100_000
# Trading costs = Alpaca's fee schedule (US stocks) + slippage. Alpaca charges no commission on stocks/ETFs and
# passes through regulatory fees. The SEC rate is reset every year: it was $0 from May 2025 to 3 Apr 2026 and
# $20.60 per $1M sold from 4 Apr 2026 (the rate used here for the whole history; earlier years were similar):
COMMISSION_PER_ORDER = 0.0       # $ per order: Alpaca is commission-free for US stocks and ETFs
SEC_FEE_RATE         = 0.0000206 # SEC Section 31 fee x sell value (sells only)
FINRA_TAF_PER_SHARE  = 0.000195  # FINRA Trading Activity Fee per share sold (sells only) ...
FINRA_TAF_MAX        = 9.79      # ... capped at $9.79 per trade
CAT_FEE_PER_SHARE    = 0.000003  # FINRA Consolidated Audit Trail fee per share (buys and sells)
# Slippage by order type (1 bp = 0.01%). Orders decided after the close are sent as market-on-open orders
# (Alpaca time_in_force="opg"): they fill in the opening auction at the official open, the same price used here,
# so only a small allowance is needed for the data vendor's open differing from the auction print.
SLIPPAGE_BPS      = 1            # market-on-open orders: entries, F-Score / time-limit / market-filter exits
SLIPPAGE_STOP_BPS = 5            # stop orders (only used if STOP_LOSS is set): a market sell into a falling price
# Take-profit sell limits fill at their limit price (or at the open if the stock gaps above it): no slippage.
RISK_FREE       = 0.0            # annual rate for Sharpe/Sortino (cash earns 0 in the simulation)
BENCHMARK       = "SPY"
FSCORE_MAX_AGE_DAYS     = 200    # F-Score treated as unknown if its fiscal period ended longer ago than this
SHARES_MAX_AGE_DAYS     = 400    # same for the share count used in market cap
MISSING_PRICE_EXIT_DAYS = 5      # close a holding at its last price after this many days without quotes

# ---------------- Checks ----------------
RUN_LOOKAHEAD_AUDIT = True
RUN_RISK_ABLATION   = True       # re-run the full walk-forward with each risk control flipped
RUN_VARIANTS        = True       # re-run the full walk-forward with other rankings / the F-Score-upgrade entry
N_BOOTSTRAP         = 2000
CSCV_BLOCKS         = 16

# ---------------- Data ----------------
SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "Your Name your.email@example.com")   # SEC requires a real contact
CACHE_DIR      = Path(os.environ.get("SMA_PF_CACHE", "data_cache"))
REPORT_DIR     = Path(os.environ.get("SMA_PF_REPORTS", "reports"))
REFRESH_DATA   = False           # True = re-download everything (e.g. to extend the test to today)

TD = 252   # trading days per year

### Chart style

In [ ]:
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
C_STRAT, C_BENCH = CATEGORICAL[0], CATEGORICAL[1]
DIVERGING  = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.labelcolor": INK2, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "font.size": 10, "figure.dpi": 110, "axes.prop_cycle": plt.cycler(color=CATEGORICAL),
})


def growth_axis(ax):
    """Log-scale y axis labelled as multiples of the starting value (1x, 2x, 3x ...)."""
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1, 1.5, 2, 3, 5, 7)))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}x"))
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())


def pct_axis(ax, axis="y", decimals=0):
    fmt = mticker.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)

## 2. Data
### 2a. Point-in-time S&P 500 universe

In [ ]:
SP500_HISTORY_URL = ("https://raw.githubusercontent.com/fja05680/sp500/master/"
                     "S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv")
SP500_CURRENT_URL = "https://raw.githubusercontent.com/fja05680/sp500/master/sp500.csv"
SEC_TICKERS_URL   = "https://www.sec.gov/files/company_tickers.json"
SEC_FACTS_URL     = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"

for sub in ("prices", "sec"):
    (CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def http_get(url, sec=False, retries=5):
    """GET with retries. SEC endpoints need a User-Agent with a contact and at most 10 requests/second."""
    if sec and "example.com" in SEC_USER_AGENT:
        raise RuntimeError("Set SEC_USER_AGENT (your name + e-mail) in the config cell: the SEC rejects anonymous requests.")
    headers = {"User-Agent": SEC_USER_AGENT if sec else "Mozilla/5.0 (research backtest)"}
    r = None
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=60)
        except requests.RequestException:
            r = None
        if r is not None and r.status_code == 200:
            if sec:
                time.sleep(0.12)
            return r
        if r is not None and r.status_code == 404:
            return None
        time.sleep(2 ** attempt)
    raise RuntimeError(f"Download failed: {url} (last status {getattr(r, 'status_code', 'n/a')})")


def cached_file(name, url, sec=False):
    path = CACHE_DIR / name
    if REFRESH_DATA or not path.exists():
        path.write_bytes(http_get(url, sec=sec).content)
    return path


# Old ticker -> today's ticker for companies that were renamed (same SEC registrant). Extend if you spot others.
TICKER_ALIASES = {
    "FB": "META", "ANTM": "ELV", "ABC": "COR", "RE": "EG", "WLTW": "WTW", "PKI": "RVTY", "FLT": "CPAY",
    "BLL": "BALL", "COG": "CTRA", "SYMC": "GEN", "NLOK": "GEN", "KORS": "CPRI", "HRS": "LHX", "UTX": "RTX",
    "BBT": "TFC", "JEC": "J", "DISCA": "WBD", "DISCK": "WBD", "TMK": "GL", "CTL": "LUMN", "FBHS": "FBIN",
    "CDAY": "DAY", "HCP": "DOC", "PEAK": "DOC", "LB": "BBWI", "BHGE": "BKR", "GPS": "GAP", "FI": "FISV",
    "MMC": "MRSH",
}
YF_TO_GICS = {"Technology": "Information Technology", "Healthcare": "Health Care", "Financial Services": "Financials",
              "Consumer Cyclical": "Consumer Discretionary", "Consumer Defensive": "Consumer Staples",
              "Basic Materials": "Materials", "Communication Services": "Communication Services",
              "Industrials": "Industrials", "Energy": "Energy", "Utilities": "Utilities", "Real Estate": "Real Estate"}


def yahoo_symbol(t):
    t = t.strip().upper().replace(".", "-")
    return TICKER_ALIASES.get(t, t)


def load_universe():
    """Membership matrix (change dates x tickers), ticker->CIK map and GICS sectors."""
    hist = pd.read_csv(cached_file("sp500_history.csv", SP500_HISTORY_URL))
    hist["date"] = pd.to_datetime(hist["date"])
    hist = hist.sort_values("date").drop_duplicates("date", keep="last")
    first = hist.loc[hist["date"] <= pd.Timestamp(PRICE_START), "date"]
    hist = hist[hist["date"] >= (first.iloc[-1] if len(first) else hist["date"].min())]
    canon = hist["tickers"].map(lambda s: ",".join(sorted({yahoo_symbol(t) for t in s.split(",") if t.strip()})))
    membership = canon.str.get_dummies(",").astype(bool)
    membership.index = pd.DatetimeIndex(hist["date"].values).astype("datetime64[ns]")

    cur = pd.read_csv(cached_file("sp500_current.csv", SP500_CURRENT_URL))
    cur["ticker"] = cur["Symbol"].astype(str).map(yahoo_symbol)
    sec_map = json.loads(cached_file("sec_company_tickers.json", SEC_TICKERS_URL, sec=True).read_text())
    cik = {str(v["ticker"]).upper().replace(".", "-"): int(v["cik_str"]) for v in sec_map.values()}
    cik.update({t: int(c) for t, c in zip(cur["ticker"], cur["CIK"]) if pd.notna(c)})
    sectors = dict(zip(cur["ticker"], cur["GICS Sector"]))
    return membership, cik, sectors


membership_raw, CIK_MAP, GICS = load_universe()
universe = sorted(membership_raw.columns)
no_cik = [t for t in universe if t not in CIK_MAP]
print(f"S&P 500 membership snapshots: {len(membership_raw):,} "
      f"({membership_raw.index.min():%Y-%m-%d} to {membership_raw.index.max():%Y-%m-%d})")
print(f"Tickers that were members at some point since {PRICE_START}: {len(universe)}; "
      f"without an SEC CIK today (mostly acquired/delisted): {len(no_cik)}")

### 2b. Daily prices (Yahoo Finance)
`Close/Open/High` are split-adjusted (used for signals and fills). `Adj Close` also includes dividends and is used for P&L, so returns are total returns.

In [ ]:
def _yf_download(batch):
    for attempt in range(3):
        try:
            return yf.download(batch, start=PRICE_START, auto_adjust=False, actions=True,
                               group_by="ticker", threads=True, progress=False)
        except Exception as e:               # rate limits / network hiccups
            print("  retrying:", e)
            time.sleep(10 * (attempt + 1))
    return None


def _save_prices(data, batch, pdir):
    """Cache each ticker of a downloaded batch; return the tickers that came back empty."""
    failed = []
    for t in batch:
        try:
            df = data[t] if isinstance(data.columns, pd.MultiIndex) else data
        except (KeyError, TypeError, AttributeError):
            df = None
        if df is None or "Close" not in df or df["Close"].dropna().empty:
            failed.append(t)
            continue
        df = df.dropna(subset=["Close"]).copy()
        df = df[df["Close"] > 0]
        df.index = pd.DatetimeIndex(pd.to_datetime(df.index)).tz_localize(None).astype("datetime64[ns]")
        df = df[~df.index.duplicated(keep="last")].sort_index()
        df.columns = [str(c) for c in df.columns]
        df.to_csv(pdir / f"{t}.csv.gz")
    return failed


def load_prices(tickers, chunk=80):
    pdir = CACHE_DIR / "prices"
    missing_file = pdir / "_no_data.json"
    no_data = set(json.loads(missing_file.read_text())) if missing_file.exists() and not REFRESH_DATA else set()
    todo = [t for t in tickers if t not in no_data and (REFRESH_DATA or not (pdir / f"{t}.csv.gz").exists())]
    if todo:
        if yf is None:
            raise ImportError("pip install yfinance")
        print(f"Downloading {len(todo)} price histories from Yahoo Finance ...")
    failed = []
    for k in range(0, len(todo), chunk):
        batch = todo[k:k + chunk]
        data = _yf_download(batch)
        failed += batch if data is None else _save_prices(data, batch, pdir)
        print(f"  {min(k + chunk, len(todo))}/{len(todo)}")
    if failed:                               # second pass in small batches: Yahoo drops symbols under load
        time.sleep(5)
        retry, failed = failed, []
        for k in range(0, len(retry), 20):
            batch = retry[k:k + 20]
            data = _yf_download(batch)
            failed += batch if data is None else _save_prices(data, batch, pdir)
    missing_file.write_text(json.dumps(sorted(no_data | set(failed))))
    frames = {}
    for t in tickers:
        f = pdir / f"{t}.csv.gz"
        if f.exists():
            df = pd.read_csv(f, index_col=0)
            df.index = pd.DatetimeIndex(pd.to_datetime(df.index)).astype("datetime64[ns]")
            frames[t] = df[~df.index.duplicated(keep="last")].sort_index()
    return frames


price_frames = load_prices([t for t in universe if t in CIK_MAP] + [BENCHMARK])
assert BENCHMARK in price_frames, f"No prices for the benchmark {BENCHMARK}"
print(f"Price histories available: {len(price_frames) - 1} of {len(universe) - len(no_cik)} tickers "
      "(Yahoo drops most delisted/acquired stocks; see Limitations)")

### 2c. Point-in-time fundamentals (SEC EDGAR XBRL) → Piotroski F-Score

Same nine tests as `Piotroski-F-Score.ipynb`, computed at every 10-Q/10-K balance-sheet date:

| Test | Rule (current period vs same period one year earlier) |
|---|---|
| ROA | Net income / total assets > 0 |
| CFO | Operating cash flow / total assets > 0 |
| ΔROA | ROA higher than a year ago |
| ACCRUAL | Operating cash flow > net income |
| ΔLEVER | Long-term debt / total assets lower than a year ago (untagged debt = 0) |
| ΔLIQUID | Current ratio higher than a year ago |
| EQ_OFFER | Diluted shares not higher than a year ago |
| ΔMARGIN | Gross margin % higher than a year ago |
| ΔTURN | Revenue / total assets higher than a year ago |

* Flows (revenue, COGS, net income, cash flow) use **trailing-twelve-month** sums by default (`FSCORE_BASIS="ttm"`), which removes seasonality. Quarter values are taken from 3-month facts, or derived from year-to-date facts (Q4 = FY − 9M; cash-flow quarters = YTD differences).
* A test whose inputs are missing scores **0**. The score can't pass on data that isn't there.
* **Availability date** = filing date of the 10-Q/10-K. The score is usable from the next trading day. Inputs first filed later (amendments) are ignored for that period, and the first-reported value is kept.

In [ ]:
FORMS = {"10-K", "10-Q", "10-K/A", "10-Q/A", "10-KT", "10-KT/A"}
FIELDS = {   # field: (list of (taxonomy, tag) in priority order, unit)
    "revenue": ([("us-gaap", "Revenues"), ("us-gaap", "RevenueFromContractWithCustomerExcludingAssessedTax"),
                 ("us-gaap", "RevenueFromContractWithCustomerIncludingAssessedTax"), ("us-gaap", "SalesRevenueNet"),
                 ("us-gaap", "SalesRevenueGoodsNet"), ("us-gaap", "SalesRevenueServicesNet")], "USD"),
    "cogs": ([("us-gaap", "CostOfRevenue"), ("us-gaap", "CostOfGoodsAndServicesSold"), ("us-gaap", "CostOfGoodsSold"),
              ("us-gaap", "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization"),
              ("us-gaap", "CostOfServices")], "USD"),
    "gross_profit": ([("us-gaap", "GrossProfit")], "USD"),
    "net_income": ([("us-gaap", "NetIncomeLoss"), ("us-gaap", "ProfitLoss"),
                    ("us-gaap", "NetIncomeLossAvailableToCommonStockholdersBasic")], "USD"),
    "ocf": ([("us-gaap", "NetCashProvidedByUsedInOperatingActivities"),
             ("us-gaap", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations")], "USD"),
    "total_assets": ([("us-gaap", "Assets")], "USD"),
    "current_assets": ([("us-gaap", "AssetsCurrent")], "USD"),
    "current_liabilities": ([("us-gaap", "LiabilitiesCurrent")], "USD"),
    "lt_debt": ([("us-gaap", "LongTermDebtNoncurrent"), ("us-gaap", "LongTermDebtAndCapitalLeaseObligations"),
                 ("us-gaap", "LongTermDebt"), ("us-gaap", "LongTermNotesPayable")], "USD"),
    "diluted_shares": ([("us-gaap", "WeightedAverageNumberOfDilutedSharesOutstanding"),
                        ("us-gaap", "WeightedAverageNumberOfShareOutstandingBasicAndDiluted")], "shares"),
    "shares_out": ([("dei", "EntityCommonStockSharesOutstanding")], "shares"),
}
FLOWS = ["revenue", "cogs", "gross_profit", "net_income", "ocf"]
STOCKS = ["total_assets", "current_assets", "current_liabilities", "lt_debt"]
TEST_NAMES = ["ROA", "CFO", "dROA", "ACCRUAL", "dLEVER", "dLIQUID", "EQ_OFFER", "dMARGIN", "dTURN"]
KEEP_KEYS = ("start", "end", "val", "filed", "form", "accn")


def load_facts(cik):
    """Needed XBRL facts for one company, cached as {taxonomy: {tag: [facts]}}."""
    path = CACHE_DIR / "sec" / f"CIK{cik:010d}.json.gz"
    if path.exists() and not REFRESH_DATA:
        with gzip.open(path, "rt") as fh:
            return json.load(fh)
    r = http_get(SEC_FACTS_URL.format(cik=cik), sec=True)
    out = {}
    if r is not None:
        facts = r.json().get("facts", {})
        for tags, unit in FIELDS.values():
            for ns, tag in tags:
                rows = facts.get(ns, {}).get(tag, {}).get("units", {}).get(unit, [])
                rows = [{k: f.get(k) for k in KEEP_KEYS} for f in rows if f.get("form") in FORMS]
                if rows:
                    out.setdefault(ns, {})[tag] = rows
    with gzip.open(path, "wt") as fh:
        json.dump(out, fh)
    return out


def _dt(x):
    return pd.to_datetime(x).astype("datetime64[ns]")


def _frame(facts, field, cutoff=None):
    """All facts for one field (every tag), optionally only those filed on/before `cutoff`."""
    tags, _ = FIELDS[field]
    rows = [(p, f.get("start"), f["end"], f["val"], f["filed"], f.get("accn"))
            for p, (ns, tag) in enumerate(tags) for f in facts.get(ns, {}).get(tag, [])]
    df = pd.DataFrame(rows, columns=["prio", "start", "end", "val", "filed", "accn"])
    if df.empty:
        return df
    for c in ("start", "end", "filed"):
        df[c] = _dt(df[c])
    df["val"] = df["val"].astype(float)
    if cutoff is not None:
        df = df[df["filed"] <= pd.Timestamp(cutoff)]
    return df


def _first_reported(df, keys):
    """One value per period: the earliest filing wins (restatements ignored), then tag priority."""
    return df.sort_values(["filed", "prio"], kind="stable").drop_duplicates(keys, keep="first")


def _durations(df):
    df = _first_reported(df.dropna(subset=["start"]), ["start", "end"]).copy()
    df["days"] = (df["end"] - df["start"]).dt.days
    return df


def _quarters(df):
    """Discrete fiscal-quarter values (index = period end) and 10-K annual values."""
    df = _durations(df)
    direct = df.loc[df["days"].between(75, 105), ["end", "val", "filed"]].assign(src=0)
    ytd = df[df["days"].between(75, 380)].sort_values(["start", "end"])
    prev = ytd.groupby("start")[["end", "val", "filed"]].shift()
    ok = (ytd["end"] - prev["end"]).dt.days.between(75, 105)
    derived = pd.DataFrame({"end": ytd.loc[ok, "end"], "val": ytd.loc[ok, "val"] - prev.loc[ok, "val"],
                            "filed": pd.concat([ytd.loc[ok, "filed"], prev.loc[ok, "filed"]], axis=1).max(axis=1),
                            "src": 1})
    q = pd.concat([direct, derived]).sort_values(["end", "filed", "src"]).drop_duplicates("end").set_index("end")
    annual = df.loc[df["days"].between(350, 380), ["end", "val", "filed"]].drop_duplicates("end").set_index("end")
    return q[["val", "filed"]].sort_index(), annual.sort_index()


def _ttm(q, annual):
    """Trailing-12-month sum of four consecutive quarters; a 10-K annual figure is used directly when available."""
    out = [annual.assign(src=0)]
    if len(q) >= 4:
        e = q.index.to_series()
        steps_ok = e.diff().dt.days.between(75, 105).astype(int).rolling(3).sum() == 3
        span_ok = (e - e.shift(3)).dt.days.between(250, 300)
        filed_day = pd.Series(q["filed"].values.astype("datetime64[D]").astype("int64"), index=q.index)
        last_filed = filed_day.rolling(4).max()          # whole days: exact in float
        roll = pd.DataFrame({"val": q["val"].rolling(4).sum(),
                             "filed": _dt(pd.to_datetime(last_filed.fillna(0).astype("int64"), unit="D")
                                          .where(last_filed.notna()))})
        out.append(roll[steps_ok & span_ok].assign(src=1))
    ttm = pd.concat(out)
    ttm.index.name = "end"
    return (ttm.reset_index().sort_values(["end", "filed", "src"]).drop_duplicates("end")
               .set_index("end")[["val", "filed"]])


def _instants(df):
    return _first_reported(df, ["end"]).set_index("end")[["val", "filed"]].sort_index()


def _avg_shares(df):
    """Weighted diluted shares per period end (first public figure; the quarter if several came out together)."""
    df = _durations(df)
    df = df[df["days"].between(75, 380)].sort_values(["end", "filed", "days"]).drop_duplicates("end")
    return df.set_index("end")[["val", "filed"]]


def _shares_pair(df, ends, known):
    """Weighted diluted shares for each period and for the same period a year earlier, both as known on the
    period's filing date. The prior-year figure comes from the latest filing up to that date (normally the
    comparative column of the same report), so stock splits in between are already restated."""
    nan = np.full(len(ends), np.nan)
    if df.empty:
        return nan, nan.copy()
    d = df.dropna(subset=["start"])
    days = (d["end"] - d["start"]).dt.days.to_numpy()
    keep = (days >= 75) & (days <= 380)
    d, days = d[keep], days[keep]
    order = np.lexsort((-d["prio"].to_numpy(), d["filed"].to_numpy()))   # by filing date, best tag last
    end = d["end"].to_numpy()[order]; filed = d["filed"].to_numpy()[order]
    val = d["val"].to_numpy(float)[order]; days = days[order]
    day = np.timedelta64(1, "D")
    cur, pri = nan.copy(), nan.copy()
    for n, (e, k) in enumerate(zip(ends.to_numpy(), np.asarray(known, dtype="datetime64[ns]"))):
        if np.isnat(k):
            continue
        public = filed <= k
        c = np.flatnonzero(public & (np.abs(end - e) <= 7 * day))
        if not c.size:
            continue
        c = c[np.lexsort((filed[c], days[c]))][0]                 # shortest period, first reported
        cur[n] = val[c]
        pe = e - 364 * day
        p = np.flatnonzero(public & (np.abs(end - pe) <= 20 * day) & (np.abs(days - days[c]) <= 20))
        if p.size:
            pri[n] = val[p[-1]]                                    # latest filing up to k
    return cur, pri


def _align(ends, s, tol_days=7):
    """Value/filing date of series `s` at each period end in `ends` (nearest within tolerance)."""
    left = pd.DataFrame({"end": ends})
    if s is None or len(s) == 0:
        return pd.DataFrame({"val": np.nan, "filed": pd.NaT}, index=ends)
    right = s.reset_index().rename(columns={"end": "k"}).sort_values("k")
    m = pd.merge_asof(left, right, left_on="end", right_on="k", direction="nearest",
                      tolerance=pd.Timedelta(days=tol_days))
    return m.set_index("end")[["val", "filed"]]


def fscore_history(facts, basis=FSCORE_BASIS, cutoff=None):
    """F-Score at every balance-sheet date with the date it became public. None if no usable data."""
    raw = {f: _frame(facts, f, cutoff) for f in FIELDS}
    if raw["total_assets"].empty:
        return None
    bs = _instants(raw["total_assets"])
    ends = pd.DatetimeIndex(bs.index.unique()).sort_values()
    cols = {"total_assets": bs.reindex(ends)}
    for f in STOCKS[1:]:
        cols[f] = _align(ends, _instants(raw[f]) if not raw[f].empty else None)
    for f in FLOWS:
        s = None
        if not raw[f].empty:
            q, ann = _quarters(raw[f])
            s = _ttm(q, ann) if basis == "ttm" else q
        cols[f] = _align(ends, s)
    known = cols["total_assets"]["filed"]               # the 10-Q/10-K for this period
    vals = pd.DataFrame({f: c["val"] for f, c in cols.items()}, index=ends).astype(float)
    for f, c in cols.items():                           # not public when the period was filed -> unknown
        late = c["filed"] > known + pd.Timedelta(days=3)
        vals.loc[late.values, f] = np.nan
    vals["lt_debt"] = vals["lt_debt"].fillna(0.0).where(vals["total_assets"].notna())
    vals["gross_margin"] = (vals["revenue"] - vals["cogs"]).fillna(vals["gross_profit"])
    sh_cur, sh_prior = _shares_pair(raw["diluted_shares"], ends, known)
    vals["diluted_shares"] = sh_cur
    vals["known"] = known

    # same period one year earlier
    prior = pd.merge_asof(pd.DataFrame({"end": ends - pd.Timedelta(days=364)}),
                          vals.rename_axis("pend").reset_index(),
                          left_on="end", right_on="pend", direction="nearest", tolerance=pd.Timedelta(days=20))
    prior.index = ends
    prior["diluted_shares"] = sh_prior                  # split-consistent comparison (see _shares_pair)
    c, p = vals, prior
    roa_c, roa_p = c["net_income"] / c["total_assets"], p["net_income"] / p["total_assets"]
    tests = pd.DataFrame({
        "ROA": roa_c > 0,
        "CFO": c["ocf"] / c["total_assets"] > 0,
        "dROA": roa_c > roa_p,
        "ACCRUAL": c["ocf"] > c["net_income"],
        "dLEVER": c["lt_debt"] / c["total_assets"] < p["lt_debt"] / p["total_assets"],
        "dLIQUID": c["current_assets"] / c["current_liabilities"] > p["current_assets"] / p["current_liabilities"],
        "EQ_OFFER": c["diluted_shares"] <= p["diluted_shares"],
        "dMARGIN": c["gross_margin"] / c["revenue"] > p["gross_margin"] / p["revenue"],
        "dTURN": c["revenue"] / c["total_assets"] > p["revenue"] / p["total_assets"],
    }, index=ends).astype(int)
    need = {"ROA": ["net_income"], "CFO": ["ocf"], "dROA": ["net_income"], "ACCRUAL": ["ocf", "net_income"],
            "dLEVER": ["lt_debt"], "dLIQUID": ["current_assets", "current_liabilities"],
            "EQ_OFFER": ["diluted_shares"], "dMARGIN": ["gross_margin", "revenue"], "dTURN": ["revenue"]}
    delta = {"dROA", "dLEVER", "dLIQUID", "EQ_OFFER", "dMARGIN", "dTURN"}
    scored = pd.DataFrame({t: c[need[t]].notna().all(axis=1) & (p[need[t]].notna().all(axis=1) if t in delta else True)
                           for t in TEST_NAMES}, index=ends)
    out = tests.copy()
    out["F"] = tests.sum(axis=1).astype(float)
    out["n_scored"] = scored.sum(axis=1)
    out["known"] = pd.concat([c["known"], p["known"]], axis=1).max(axis=1)
    out = out[p["total_assets"].notna().values & c["known"].notna().values]   # need a prior-year period
    out.index.name = "end"
    return out.reset_index()


def shares_history(facts, cutoff=None):
    """Share-count records from 10-Q/10-K filings: cover-page shares outstanding (all classes summed) and weighted
    diluted shares. Both are kept; on each day the latest cover-page figure is used, and the diluted figure only
    when no cover-page figure is known yet (choosing per day, not per company, keeps this point-in-time)."""
    out = {}
    cover = _frame(facts, "shares_out", cutoff)
    if not cover.empty:
        cover = (cover.groupby(["accn", "end"], as_index=False, dropna=False)
                      .agg(val=("val", lambda s: s.drop_duplicates().sum()), filed=("filed", "min"))
                      .sort_values("filed").drop_duplicates("end"))
        out["cover"] = cover[["end", "val", "filed"]].sort_values(["filed", "end"])
    dil = _frame(facts, "diluted_shares", cutoff)
    if not dil.empty:
        out["diluted"] = _avg_shares(dil).reset_index().sort_values(["filed", "end"])
    return out or None


def compute_fundamentals(tickers, cutoff=None, verbose=True):
    out, t0 = {}, time.time()
    for n, t in enumerate(tickers, 1):
        facts = load_facts(CIK_MAP[t])
        fs = fscore_history(facts, cutoff=cutoff)
        sh = shares_history(facts, cutoff=cutoff)
        if fs is not None and len(fs) and sh:
            out[t] = (fs, sh)
        if verbose and n % 100 == 0:
            print(f"  {n}/{len(tickers)} companies ({time.time() - t0:.0f}s)")
    return out

### 2d. One line per company, sectors, fundamentals
Share classes of the same company (GOOG/GOOGL, FOX/FOXA, NWS/NWSA) are merged into one line so the book can't hold the same company twice.

In [ ]:
by_cik = {}
for t in price_frames:
    if t != BENCHMARK:
        by_cik.setdefault(CIK_MAP[t], []).append(t)
PRIMARY = {cik: sorted(ts, key=lambda t: (-len(price_frames[t]), t))[0] for cik, ts in by_cik.items()}

# membership of a company = member under any of its tickers
membership = pd.DataFrame({PRIMARY[cik]: membership_raw[[t for t in membership_raw.columns if CIK_MAP.get(t) == cik]].any(axis=1)
                           for cik in PRIMARY})


def load_sectors(tickers):
    """GICS sector from the constituents file; Yahoo's sector (mapped to GICS names) for former members."""
    path = CACHE_DIR / "sectors_yf.json"
    cache = json.loads(path.read_text()) if path.exists() else {}
    lookups = [t for t in tickers if not any(x in GICS for x in by_cik[CIK_MAP[t]]) and t not in cache]
    if lookups and yf is not None:
        print(f"Looking up {len(lookups)} sectors for former S&P 500 members on Yahoo ...")
    out = {}
    for t in tickers:
        group = by_cik[CIK_MAP[t]]
        g = next((GICS[x] for x in group if x in GICS), None)
        if g is None:
            if t not in cache and yf is not None:
                try:
                    cache[t] = yf.Ticker(t).info.get("sector")
                except Exception:
                    cache[t] = None
            g = YF_TO_GICS.get(cache.get(t) or "", "Unknown")
        out[t] = g
    path.write_text(json.dumps(cache))
    return out


companies = sorted(PRIMARY.values())
print(f"Companies with prices: {len(companies)}. Loading SEC fundamentals "
      "(first run downloads one file per company, ~10/second) ...")
FUND = compute_fundamentals(companies)
SECTOR = load_sectors(list(FUND))
print(f"Companies with a usable F-Score history: {len(FUND)}")
display(pd.Series(SECTOR).loc[list(FUND)].value_counts().rename("companies").to_frame().T)

### 2e. Daily feature matrices
Everything is aligned on the benchmark's trading calendar. For each day *t* the backtest reads row *t−1* (yesterday's close) to decide, and row *t* to fill orders.

In [ ]:
def split_factor_after(splits, when):
    """Product of split ratios with ex-date after each date in `when`: restates old share counts in today's units."""
    if splits is None or len(splits) == 0:
        return np.ones(len(when))
    sd, ratio = splits.index.values, splits.values.astype(float)
    cum_after = np.cumprod(ratio[::-1])[::-1]
    k = np.searchsorted(sd, np.asarray(when, dtype="datetime64[ns]"), side="right")
    out = np.ones(len(when))
    ok = k < len(sd)
    out[ok] = cum_after[k[ok]]
    return out


def to_daily(values, known, dates, period_end=None, max_age=None):
    """Point-in-time daily series: a record is usable from the first trading day strictly after it was filed."""
    values, known = np.asarray(values, float), np.asarray(known, dtype="datetime64[ns]")
    pos = dates.searchsorted(known, side="right")
    ok = pos < len(dates)
    p_ok, v_ok = pos[ok], values[ok]
    rev = len(p_ok) - 1 - np.unique(p_ok[::-1], return_index=True)[1]   # last record per day (inputs sorted by filed, end)
    arr = np.full(len(dates), np.nan)
    arr[p_ok[rev]] = v_ok[rev]
    arr = pd.Series(arr).ffill().to_numpy(copy=True)
    if period_end is not None:
        end_ns = np.asarray(period_end, dtype="datetime64[ns]").astype("int64").astype(float)[ok]
        e = np.full(len(dates), np.nan)
        e[p_ok[rev]] = end_ns[rev]
        e = pd.Series(e).ffill().to_numpy()
        age = (dates.values.astype("int64") - e) / 86_400e9
        arr[~(age <= max_age)] = np.nan
    return arr


def build_features(frames, fund, membership, cutoff=None):
    bench = frames[BENCHMARK]
    dates = pd.DatetimeIndex(bench.index)
    dates = dates[dates >= pd.Timestamp(PRICE_START)]
    if BACKTEST_END:
        dates = dates[dates <= pd.Timestamp(BACKTEST_END)]
    if cutoff is not None:
        dates = dates[dates <= pd.Timestamp(cutoff)]
    dates = dates.astype("datetime64[ns]")
    tick = [t for t in sorted(fund) if t in frames]
    panel = lambda col: pd.DataFrame({t: frames[t][col] for t in tick}).reindex(dates)
    O, H, L, C, AC = panel("Open"), panel("High"), panel("Low"), panel("Close"), panel("Adj Close")
    O = O.where(O > 0)

    fsc, fsp, mcap = {}, {}, {}
    for t in tick:
        fs, sh = fund[t]
        fs = fs.sort_values(["known", "end"])
        fs = fs[fs["end"] >= fs["end"].cummax()]          # a late filing for an older period never overrides a newer one
        fsc[t] = to_daily(fs["F"], fs["known"], dates, fs["end"], FSCORE_MAX_AGE_DAYS)
        fsp[t] = to_daily(fs["F"].shift(1), fs["known"], dates, fs["end"], FSCORE_MAX_AGE_DAYS)  # score before it
        splits = frames[t]["Stock Splits"] if "Stock Splits" in frames[t] else None
        if splits is not None:
            splits = splits[splits.fillna(0) > 0]
        daily = {}
        for src, rec in sh.items():
            adj_shares = rec["val"].to_numpy(float) * split_factor_after(splits, rec["end"])
            daily[src] = to_daily(adj_shares, rec["filed"], dates, rec["end"], SHARES_MAX_AGE_DAYS)
        shares = daily.get("cover", np.full(len(dates), np.nan))
        mcap[t] = np.where(np.isfinite(shares), shares, daily.get("diluted", np.nan))
    FS = pd.DataFrame(fsc, index=dates)[tick].to_numpy()
    FSP = pd.DataFrame(fsp, index=dates)[tick].to_numpy()
    MC = C.to_numpy() * pd.DataFrame(mcap, index=dates)[tick].to_numpy()

    mpos = membership.index.searchsorted(dates, side="right") - 1
    MEM = membership.reindex(columns=tick, fill_value=False).to_numpy()[np.clip(mpos, 0, None)]
    MEM[mpos < 0] = False

    sector = [SECTOR.get(t, "Unknown") for t in tick]
    names = sorted(set(sector))
    code = np.array([names.index(s) for s in sector])
    excl = np.array([s in EXCLUDE_SECTORS for s in sector])

    min_obs = lambda w: max(int(w * 0.95), 1)
    sma = {w: C.rolling(w, min_periods=min_obs(w)).mean().to_numpy()
           for w in sorted(set(PARAM_GRID.get("sma_fast", [])) | {SMA_FAST, SMA_SLOW})}
    Cn = C.to_numpy()
    AF = (AC / C).ffill().to_numpy()
    F = dict(dates=dates, tickers=np.array(tick), open=O.to_numpy(), high=H.to_numpy(), low=L.to_numpy(), close=Cn,
             adj_ff=AC.ffill().to_numpy(), af=AF, fscore=FS, fscore_prev=FSP, mcap=MC, member=MEM, sector_code=code,
             sector_names=names, excluded=excl, sma200=sma[SMA_SLOW])
    with np.errstate(invalid="ignore"):
        F["base"] = MEM & np.isfinite(Cn) & (Cn < sma[SMA_SLOW]) & np.isfinite(MC) & (MC > 0) & ~excl[None, :]
        F["above"] = {w: Cn > sma[w] for w in sma}
    F["sma"] = sma
    rets = AC.pct_change(fill_method=None)
    vol = rets.rolling(VOL_LOOKBACK, min_periods=int(VOL_LOOKBACK * 0.8)).std() * np.sqrt(TD)
    F["vol"] = vol.to_numpy()
    # median volatility of all S&P 500 members with prices that day (not only stocks with an F-Score somewhere in
    # the data, which would depend on future filings)
    all_tick = [t for t in membership.columns if t in frames and t != BENCHMARK]
    AC_all = pd.DataFrame({t: frames[t]["Adj Close"] for t in all_tick}).reindex(dates)
    vol_all = AC_all.pct_change(fill_method=None).rolling(VOL_LOOKBACK, min_periods=int(VOL_LOOKBACK * 0.8)).std()
    mem_all = membership.reindex(columns=all_tick, fill_value=False).to_numpy()[np.clip(mpos, 0, None)]
    mem_all[mpos < 0] = False
    F["vol_ref"] = vol_all.where(mem_all).median(axis=1).to_numpy() * np.sqrt(TD)
    bench_close = bench["Close"].reindex(dates)
    bench_sma = bench_close.rolling(MARKET_FILTER_SMA, min_periods=min_obs(MARKET_FILTER_SMA)).mean()
    F["market_ok"] = (bench_close >= bench_sma).to_numpy() | bench_sma.isna().to_numpy()   # True = risk-on
    bench_adj = bench["Adj Close"].reindex(dates)
    F["bench_ret"] = bench_adj.pct_change()
    return F


FEAT = build_features(price_frames, FUND, membership)
print(f"Feature matrices: {len(FEAT['dates']):,} days x {len(FEAT['tickers'])} stocks "
      f"({FEAT['dates'][0]:%Y-%m-%d} to {FEAT['dates'][-1]:%Y-%m-%d})")

In [ ]:
# Coverage check: how many stocks pass each filter over time
with np.errstate(invalid="ignore"):
    cov = pd.DataFrame({
        "S&P 500 members with data": (FEAT["member"] & np.isfinite(FEAT["close"])).sum(1),
        "... with an F-Score": (FEAT["member"] & np.isfinite(FEAT["fscore"])).sum(1),
        f"... F-Score >= {F_MIN}": (FEAT["member"] & (FEAT["fscore"] >= F_MIN)).sum(1),
        f"... and SMA{SMA_FAST} < close < SMA{SMA_SLOW} (entry candidates)":
            (FEAT["base"] & FEAT["above"][SMA_FAST] & (FEAT["fscore"] >= F_MIN)).sum(1),
    }, index=FEAT["dates"])
cov = cov[cov.index >= BACKTEST_START]
fig, ax = plt.subplots(figsize=(11, 4))
for k, col in enumerate(cov.columns):
    series = cov[col].rolling(5).mean() if k == 3 else cov[col]
    ax.plot(series.index, series, lw=1.6 if k < 3 else 1.2, color=CATEGORICAL[k], label=col)
ax.set_title("Universe coverage: stocks passing each filter")
ax.set_ylabel("stocks")
ax.set_ylim(0, None)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
plt.tight_layout(); plt.show()
display(cov.groupby(cov.index.year).mean().round(0).astype(int).T)

## 3. Backtest engine
Daily event loop. For each trading day *t*:
1. **At the open**, sell holdings whose F-Score (known at *t−1*) is below `F_EXIT` or has gone stale, and holdings that have reached `MAX_HOLD_DAYS` trading days.
2. **At the open**, fill free slots. Candidates are stocks that pass all filters at *t−1*, ranked by `RANK_BY` (market cap by default), skipping sectors that already hold `max_per_sector` stocks. Each new position gets `equity / max_positions`. With `sizing="inverse_vol"`, that amount is multiplied by *(median member volatility ÷ the stock's volatility)*, kept within `VOL_WEIGHT_BOUNDS`. Volatility is measured over the `VOL_LOOKBACK` days up to *t−1*, and entries stop when cash runs out. An entry is skipped if the stock opens at or above its take-profit level.
3. **Intraday exits**: a sell limit at the *t−1* SMA200 (take profit) and a stop at `entry price × (1 − STOP_LOSS)`. A gap through either level fills at the open. If one day's range touches both levels, the **stop is assumed to fill first** (worst case, since daily bars don't show which came first).

**Market filter**: if the benchmark closed below its 200-day SMA at *t−1*, step 2 is skipped (`"no_new_entries"`). With `"liquidate"`, all positions are also sold at the open.
4. **At the close**, mark to market with dividend-adjusted prices.

Costs: Alpaca's fees on every fill, plus slippage that depends on the order type. Market-on-open orders (entries and open exits) pay `SLIPPAGE_BPS`, take-profit limits pay none, and stops pay `SLIPPAGE_STOP_BPS`. That means $0 commission, the CAT fee on buys and sells, and the SEC fee and FINRA TAF on sells. Per-share fees use split-adjusted share counts, which slightly overstates them for stocks that split later (conservative). Idle cash earns nothing. When walk-forward switches parameters, open positions are kept and the new rules apply from the next decision. For example, tightening the sector cap or the portfolio size doesn't force a sale; there is simply no buying in that sector, or at all, until holdings fall back under the new limit. Positions are sized once, at entry, and are not rebalanced.

In [ ]:
def default_params(**kw):
    p = dict(sma_fast=SMA_FAST, max_per_sector=MAX_PER_SECTOR, f_min=F_MIN, max_positions=MAX_POSITIONS, sizing=SIZING)
    p.update(kw)
    return p


def label(p):
    cap = f"<={p['max_per_sector']}/sector" if p["max_per_sector"] else "no sector cap"
    s = f"SMA{p['sma_fast']} | {cap} | {p['max_positions']} pos | {'inv-vol' if p['sizing'] == 'inverse_vol' else 'equal'}"
    if p["f_min"] != F_MIN:
        s += f" | F>={p['f_min']}"
    return s


def buy_fees(shares):
    return COMMISSION_PER_ORDER + CAT_FEE_PER_SHARE * shares


def sell_fees(value, shares):
    return (COMMISSION_PER_ORDER + SEC_FEE_RATE * value + min(FINRA_TAF_PER_SHARE * shares, FINRA_TAF_MAX)
            + CAT_FEE_PER_SHARE * shares)


def run_backtest(F, schedule, start=None, end=None, slippage_bps=SLIPPAGE_BPS, tp_mode=TP_MODE, capital=INITIAL_CAPITAL,
                 stop_loss=STOP_LOSS, market_filter=MARKET_FILTER, max_hold=MAX_HOLD_DAYS, rank_by=RANK_BY,
                 entry_trigger=ENTRY_TRIGGER):
    """schedule: a params dict, or a list of (first_date, params) for walk-forward parameter switching."""
    if isinstance(schedule, dict):
        schedule = [(None, schedule)]
    dates = F["dates"]
    i0 = max(1, int(dates.searchsorted(pd.Timestamp(start or BACKTEST_START))))
    i1 = len(dates) if end is None else int(dates.searchsorted(pd.Timestamp(end), side="right"))
    sw = [0 if d is None else int(dates.searchsorted(pd.Timestamp(d))) for d, _ in schedule]
    O, H, L, C, ACF, AF, S200, FS, MC = (F[k] for k in ("open", "high", "low", "close", "adj_ff", "af", "sma200",
                                                         "fscore", "mcap"))
    MKT = F["market_ok"]
    sec, base, tick, VOL, VREF = F["sector_code"], F["base"], F["tickers"], F["vol"], F["vol_ref"]
    slip, slip_stop = slippage_bps / 1e4, SLIPPAGE_STOP_BPS / 1e4
    costs = {"fees": 0.0, "slippage": 0.0}
    cash, pos, trades = float(capital), {}, []
    n = i1 - i0
    equity, npos, invested = np.full(n, np.nan), np.zeros(n, int), np.zeros(n)
    holdings, traded = [], 0.0
    k = 0

    def close_pos(j, i, px, af, reason, slip=slip):
        nonlocal cash, traded
        p = pos.pop(j)
        value = p["qty"] * px * (1 - slip) * af              # sold slightly below the quoted price
        fee = sell_fees(value, p["shares"])
        proceeds = value - fee
        if reason.startswith("take profit") and proceeds <= p["alloc"]:
            reason = "SMA200 exit at a loss"           # the SMA200 fell below the entry price while held
        cash += proceeds
        traded += value
        costs["fees"] += fee
        costs["slippage"] += p["qty"] * px * slip * af
        trades.append(dict(ticker=tick[j], sector=F["sector_names"][sec[j]], entry_date=dates[p["i"]],
                           exit_date=dates[i], entry_px=p["px"], exit_px=px, ret=proceeds / p["alloc"] - 1,
                           pnl=proceeds - p["alloc"], fees=p["fee"] + fee, days=(dates[i] - dates[p["i"]]).days,
                           bars=i - p["i"], reason=reason, f_entry=p["f"], mcap_bn=p["mcap"] / 1e9, weight=p["weight"],
                           config=p["cfg"]))

    for t_, i in enumerate(range(i0, i1)):
        while k + 1 < len(schedule) and i >= sw[k + 1]:
            k += 1
        prm = schedule[k][1]
        fmin, maxp, cap = prm["f_min"], prm["max_positions"], prm["max_per_sector"]
        inv_vol = prm["sizing"] == "inverse_vol"
        pr = i - 1
        eq_prev = equity[t_ - 1] if t_ > 0 else capital

        # 1) exits at the open, decided at yesterday's close
        for j in list(pos):
            if not np.isfinite(O[i, j]):
                pos[j]["miss"] += 1
                if pos[j]["miss"] >= MISSING_PRICE_EXIT_DAYS:
                    close_pos(j, i, ACF[i, j] / AF[i, j], AF[i, j], "no price data", slip=0.0)
                continue
            pos[j]["miss"] = 0
            if market_filter == "liquidate" and not MKT[pr]:
                close_pos(j, i, O[i, j], AF[i, j], "market filter")
            elif not FS[pr, j] >= F_EXIT:
                close_pos(j, i, O[i, j], AF[i, j], "F-Score below min")
            elif max_hold and i - pos[j]["i"] >= max_hold:
                close_pos(j, i, O[i, j], AF[i, j], "time limit")
            elif tp_mode == "close" and C[pr, j] >= S200[pr, j]:
                close_pos(j, i, O[i, j], AF[i, j], "take profit (SMA200)")

        # 2) entries at the open
        slots = maxp - len(pos)
        if slots > 0 and (market_filter is None or MKT[pr]):
            with np.errstate(invalid="ignore"):
                ok = base[pr] & F["above"][prm["sma_fast"]][pr] & (FS[pr] >= fmin)
                if entry_trigger == "upgrade":                 # latest filing moved the score up from below 7
                    ok &= F["fscore_prev"][pr] < F_EXIT
                cand = np.flatnonzero(ok)
            if cand.size:
                if rank_by == "upside":                         # furthest below the SMA200 target first
                    cand = cand[np.argsort(-(S200[pr, cand] / C[pr, cand]), kind="stable")]
                elif rank_by == "fscore":                       # highest F-Score first, then market cap
                    cand = cand[np.lexsort((-MC[pr, cand], -FS[pr, cand]))]
                else:
                    cand = cand[np.argsort(-MC[pr, cand], kind="stable")]
                used = Counter(sec[j] for j in pos)
                budget = eq_prev / maxp
                for j in cand:
                    if slots == 0:
                        break
                    if j in pos or (cap and used[sec[j]] >= cap):
                        continue
                    o = O[i, j]
                    if not (np.isfinite(o) and np.isfinite(AF[i, j])) or o >= S200[pr, j]:
                        continue                     # no quote, or already through the take-profit level
                    w = 1.0
                    if inv_vol and VOL[pr, j] > 0 and np.isfinite(VREF[pr]):
                        w = float(np.clip(VREF[pr] / VOL[pr, j], *VOL_WEIGHT_BOUNDS))
                    alloc = min(budget * w, cash)
                    if alloc < 0.02 * budget:
                        break
                    fill = o * (1 + slip)                            # bought slightly above the quoted open
                    fee = buy_fees(alloc / fill)
                    qty = (alloc - fee) / (fill * AF[i, j])
                    cash -= alloc
                    traded += alloc - fee
                    costs["fees"] += fee
                    costs["slippage"] += (alloc - fee) * slip / (1 + slip)
                    pos[j] = dict(qty=qty, shares=(alloc - fee) / fill, i=i, px=o, alloc=alloc, fee=fee,
                                  weight=alloc / eq_prev, f=FS[pr, j],
                                  mcap=MC[pr, j],
                                  cfg=label(prm), miss=0, stop=o * (1 - stop_loss) if stop_loss else -np.inf)
                    used[sec[j]] += 1
                    slots -= 1

        # 3) intraday exits: take-profit limit at yesterday's SMA200 and the stop loss (stop first if both touched)
        for j in list(pos):
            o, h, lo, stop = O[i, j], H[i, j], L[i, j], pos[j]["stop"]
            tp = S200[pr, j] if tp_mode == "limit" else np.inf
            if not np.isfinite(o):
                continue
            if o >= tp:
                close_pos(j, i, o, AF[i, j], "take profit (SMA200)", slip=0.0)      # resting limit fills in the auction
            elif o <= stop:
                close_pos(j, i, o, AF[i, j], "stop loss", slip=slip_stop)
            elif lo <= stop:
                close_pos(j, i, stop, AF[i, j], "stop loss", slip=slip_stop)
            elif h >= tp:
                close_pos(j, i, tp, AF[i, j], "take profit (SMA200)", slip=0.0)

        # 4) mark to market
        mv = sum(p["qty"] * ACF[i, j] for j, p in pos.items())
        equity[t_] = cash + mv
        npos[t_], invested[t_] = len(pos), mv
        holdings.append(tuple(pos))

    idx = dates[i0:i1]
    eq = pd.Series(equity, idx)
    open_pos = [dict(ticker=tick[j], sector=F["sector_names"][sec[j]], entry_date=dates[p["i"]], entry_px=p["px"],
                     last_close=C[i1 - 1, j], unrealised=p["qty"] * ACF[i1 - 1, j] / p["alloc"] - 1,
                     days=(dates[i1 - 1] - dates[p["i"]]).days, f_now=FS[i1 - 1, j], config=p["cfg"])
                for j, p in pos.items()]
    rets = eq.pct_change()
    rets.iloc[0] = eq.iloc[0] / capital - 1
    return dict(equity=eq, returns=rets, trades=pd.DataFrame(trades), open=pd.DataFrame(open_pos),
                n_pos=pd.Series(npos, idx), exposure=pd.Series(invested, idx) / eq, holdings=holdings,
                risk_on=pd.Series(MKT[i0 - 1:i1 - 1], idx),
                turnover=traded / eq.mean() / (len(idx) / TD), schedule=schedule,
                fees=costs["fees"], slippage=costs["slippage"])

## 4. Performance analytics

In [ ]:
def max_dd_duration(eq):
    peak = eq.cummax()
    under = eq < peak
    longest, start = pd.Timedelta(0), None
    for d, u in under.items():
        if u and start is None:
            start = d
        if not u and start is not None:
            longest, start = max(longest, d - start), None
    if start is not None:
        longest = max(longest, eq.index[-1] - start)
    return longest.days


def perf_stats(r, bench=None, rf=RISK_FREE):
    r = r.dropna()
    if len(r) < 2 or r.std() == 0:
        return {}
    eq = (1 + r).cumprod()
    yrs = len(r) / TD
    ex = r - rf / TD
    dd = eq / eq.cummax() - 1
    downside = np.sqrt((np.minimum(ex, 0) ** 2).mean()) * np.sqrt(TD)
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    monthly = (1 + r).groupby([r.index.year, r.index.month]).prod() - 1
    s = {"Total return": eq.iloc[-1] - 1, "CAGR": cagr, "Volatility (ann.)": r.std() * np.sqrt(TD),
         "Sharpe": ex.mean() / r.std() * np.sqrt(TD), "Sortino": ex.mean() * TD / downside if downside > 0 else np.nan,
         "Max drawdown": dd.min(), "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
         "Longest drawdown (days)": max_dd_duration(eq), "Ulcer index": np.sqrt((dd ** 2).mean()),
         "Skew": stats.skew(r), "Excess kurtosis": stats.kurtosis(r),
         "VaR 95% (daily)": np.percentile(r, 5), "CVaR 95% (daily)": r[r <= np.percentile(r, 5)].mean(),
         "Best month": monthly.max(), "Worst month": monthly.min(), "% positive months": (monthly > 0).mean()}
    if bench is not None:
        b = bench.reindex(r.index).fillna(0)
        beta = np.cov(r, b)[0, 1] / b.var()
        active = r - b
        s.update({"Beta": beta, "Alpha (ann.)": (r.mean() - beta * b.mean()) * TD, "Correlation": r.corr(b),
                  "Tracking error": active.std() * np.sqrt(TD),
                  "Information ratio": active.mean() / active.std() * np.sqrt(TD)})
    return s


def trade_stats(tr):
    if tr is None or tr.empty:
        return {}
    w, l = tr[tr["ret"] > 0], tr[tr["ret"] <= 0]
    streak = best = 0
    for x in tr.sort_values("exit_date")["ret"]:
        streak = streak + 1 if x <= 0 else 0
        best = max(best, streak)
    return {"Trades": len(tr), "Win rate": len(w) / len(tr), "Avg trade": tr["ret"].mean(),
            "Median trade": tr["ret"].median(), "Avg win": w["ret"].mean(), "Avg loss": l["ret"].mean(),
            "Payoff ratio": w["ret"].mean() / abs(l["ret"].mean()) if len(l) and len(w) else np.nan,
            "Profit factor": w["pnl"].sum() / abs(l["pnl"].sum()) if len(l) and l["pnl"].sum() else np.nan,
            "Best trade": tr["ret"].max(), "Worst trade": tr["ret"].min(),
            "Avg holding (days)": tr["days"].mean(), "Median holding (days)": tr["days"].median(),
            "Max consecutive losses": best}


def run_summary(res, bench):
    s = perf_stats(res["returns"], bench)
    s.update({"Avg positions": res["n_pos"].mean(), "Avg exposure": res["exposure"].mean(),
              "Turnover (x/yr)": res["turnover"], "Alpaca fees paid ($)": res["fees"],
              "Slippage paid ($)": res["slippage"]})
    s.update(trade_stats(res["trades"]))
    return s


PCT_ROWS = {"Total return", "CAGR", "Volatility (ann.)", "Max drawdown", "Ulcer index", "VaR 95% (daily)",
            "CVaR 95% (daily)", "Best month", "Worst month", "% positive months", "Alpha (ann.)", "Tracking error",
            "Avg exposure", "Win rate", "Avg trade", "Median trade", "Avg win", "Avg loss", "Best trade", "Worst trade"}


def fmt_table(df):
    """Format a metrics table (rows = metric names) for display."""
    def f(row):
        if row.name in PCT_ROWS:
            return row.map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
        if row.name in {"Trades", "Longest drawdown (days)", "Max consecutive losses", "Longest trade (days)",
                        "Alpaca fees paid ($)", "Slippage paid ($)"}:
            return row.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")
        if row.name in {"Drawdowns worse than -20%"}:
            return row.map(lambda v: "" if pd.isna(v) else f"{v:.0f}")
        return row.map(lambda v: "" if pd.isna(v) else f"{v:,.2f}")
    return df.apply(f, axis=1)


def window_metrics(Rw, bench):
    """Scorecard metrics for every column of Rw (daily returns over one window), same definitions as perf_stats."""
    X = Rw.to_numpy(float)
    n = len(X)
    b = bench.reindex(Rw.index).fillna(0).to_numpy(float)
    with np.errstate(all="ignore"):
        eq = np.cumprod(1 + X, axis=0)
        mdd = (eq / np.maximum.accumulate(eq, axis=0) - 1).min(axis=0)
        ex = X - RISK_FREE / TD
        sd = X.std(axis=0, ddof=1)
        down = np.sqrt((np.minimum(ex, 0) ** 2).mean(axis=0)) * np.sqrt(TD)
        cagr = eq[-1] ** (TD / n) - 1
        m = pd.DataFrame({"Total return": eq[-1] - 1, "Max drawdown": mdd, "Sharpe": ex.mean(0) / sd * np.sqrt(TD),
                          "Calmar": np.where(mdd < 0, cagr / np.abs(mdd), np.nan),
                          "Beta": ((X - X.mean(0)) * (b - b.mean())[:, None]).sum(0) / (n - 1) / b.var(ddof=1),
                          "Sortino": ex.mean(0) * TD / down}, index=Rw.columns)
    m.loc[~(sd > 0) | (n < 60)] = np.nan               # never traded, or window too short: not eligible
    return m


def score_window(Rw, bench):
    """SCORECARD points for every configuration over one window. Returns (points, metrics, metrics won)."""
    m = window_metrics(Rw, bench)
    pts = pd.Series(0.0, index=m.index)
    won = {lab: [] for lab in m.index}
    for metric, weight, how in SCORECARD:
        col = m[metric]
        if col.notna().any():
            best = col.max() if how == "max" else col.min()
            for lab in col.index[np.isclose(col, best, rtol=1e-10, atol=1e-12)]:
                pts[lab] += weight
                won[lab].append(metric)
    return pts, m, won


def pick_config(Rw, bench):
    """Configuration to trade next: most points (or best WF_OBJECTIVE metric), ties broken by Sharpe."""
    pts, m, won = score_window(Rw, bench)
    first = pts if WF_OBJECTIVE == "points" else m[{"sharpe": "Sharpe", "sortino": "Sortino", "calmar": "Calmar",
                                                     "cagr": "Total return"}[WF_OBJECTIVE]]
    key = pd.DataFrame({"first": first.fillna(-np.inf), "sharpe": m["Sharpe"].fillna(-np.inf)})
    best = key.sort_values(["first", "sharpe"], ascending=False, kind="stable").index[0]
    return best, pts, m, won

## 5. Full-sample grid (in-sample, for information only)
Every configuration run over the whole period. **These results are not used to pick parameters.** Picking the best row here would be in-sample optimisation. They feed the overfitting diagnostics in section 8 and show how sensitive the strategy is to each parameter. The full table is saved to `reports/grid_full_sample.csv`.

In [ ]:
keys = list(PARAM_GRID)
CONFIGS = [default_params(**dict(zip(keys, v))) for v in itertools.product(*PARAM_GRID.values())]
CONFIGS = [p for p in CONFIGS                                  # a cap >= portfolio size never binds: same as no cap
           if not (p["max_per_sector"] and p["max_per_sector"] >= p["max_positions"])]
LABELS = [label(p) for p in CONFIGS]
BENCH_R = FEAT["bench_ret"]

t0 = time.time()
GRID = {lab: run_backtest(FEAT, p) for lab, p in zip(LABELS, CONFIGS)}
print(f"{len(GRID)} full-sample backtests in {time.time() - t0:.0f}s")
if all(len(res["trades"]) == 0 and len(res["open"]) == 0 for res in GRID.values()):
    raise RuntimeError("No trades in any configuration. Check the coverage chart in section 2e "
                       "(prices, F-Scores and filters) before going further.")
R = pd.DataFrame({lab: res["returns"] for lab, res in GRID.items()})

grid_tbl = pd.DataFrame({lab: run_summary(res, BENCH_R) for lab, res in GRID.items()})
grid_tbl[f"{BENCHMARK} (buy & hold)"] = pd.Series(perf_stats(BENCH_R.reindex(R.index)))
show_rows = ["CAGR", "Volatility (ann.)", "Sharpe", "Sortino", "Max drawdown", "Calmar", "Alpha (ann.)", "Beta",
             "Trades", "Win rate", "Profit factor", "Avg holding (days)", "Avg positions", "Turnover (x/yr)"]
by_sharpe = grid_tbl.loc["Sharpe", LABELS].sort_values(ascending=False)
default_lab = label(default_params())
top = list(by_sharpe.index[:15]) + ([default_lab] if default_lab in LABELS and default_lab not in by_sharpe.index[:15] else [])
display(Markdown(f"**Full sample {R.index[0]:%Y-%m-%d} to {R.index[-1]:%Y-%m-%d}, Alpaca fees + {SLIPPAGE_BPS} bp slippage on open orders: "
                 f"top 15 of {len(LABELS)} configurations by Sharpe** (plus your original rules if not among them)"))
display(fmt_table(grid_tbl[top + [f"{BENCHMARK} (buy & hold)"]].reindex(show_rows)).T)

FULL_PTS, FULL_M, FULL_WON = score_window(R, BENCH_R)
board = FULL_M.assign(points=FULL_PTS, won=pd.Series({k: ", ".join(v) for k, v in FULL_WON.items()}))
board = board[board["points"] > 0].sort_values(["points", "Sharpe"], ascending=False)
display(Markdown(f"**Scorecard over the full sample** (out of {sum(w for _, w, _ in SCORECARD):g} points; for information "
                 "only, since the walk-forward scores each in-sample window separately)"))
display(board[["points", "won", "Total return", "Max drawdown", "Sharpe", "Calmar", "Beta", "Sortino"]].style
        .format({"points": "{:.1f}", "Total return": "{:.1%}", "Max drawdown": "{:.1%}", "Sharpe": "{:.2f}",
                 "Calmar": "{:.2f}", "Beta": "{:.2f}", "Sortino": "{:.2f}"}))

In [ ]:
def fmt_value(key, v):
    return {"sma_fast": f"SMA{v}", "max_per_sector": f"{v} per sector" if v else "no limit",
            "max_positions": f"{v} stocks", "sizing": {"equal": "equal weight", "inverse_vol": "inverse vol"}.get(v, v),
            "f_min": f"F >= {v}"}.get(key, str(v))


PARAM_TITLES = {"sma_fast": "Lower trend bound", "max_per_sector": "Max stocks per sector",
                "max_positions": "Portfolio size", "sizing": "Position sizing", "f_min": "Min F-Score to buy"}
varied = [k for k in PARAM_GRID if len(PARAM_GRID[k]) > 1]
cfg_tbl = pd.DataFrame([{k: fmt_value(k, p[k]) for k in varied} | {"Sharpe": grid_tbl.loc["Sharpe", l]}
                        for p, l in zip(CONFIGS, LABELS)])

# Parameter sensitivity: Sharpe of every configuration, grouped by one parameter at a time
if varied:
    fig, axes = plt.subplots(1, len(varied), figsize=(2.7 * len(varied), 3.8), sharey=True, squeeze=False)
    for ax, key in zip(axes[0], varied):
        names = [fmt_value(key, v) for v in PARAM_GRID[key]]
        for x, name in enumerate(names):
            sub = cfg_tbl.loc[cfg_tbl[key] == name, "Sharpe"].dropna()
            ax.scatter(np.full(len(sub), x) + np.linspace(-0.15, 0.15, len(sub)), sub, s=14, color=C_STRAT,
                       alpha=0.4, lw=0)
            if len(sub):
                ax.plot([x - 0.28, x + 0.28], [sub.median()] * 2, color=INK, lw=2)
        ax.set_xticks(range(len(names)), names, rotation=30, ha="right")
        ax.set_xlim(-0.6, len(names) - 0.4)
        ax.set_title(PARAM_TITLES.get(key, key), fontsize=10)
        ax.grid(axis="x", visible=False)
    axes[0][0].set_ylabel("Sharpe (full sample)")
    fig.suptitle("Parameter sensitivity: each dot is one configuration, the bar is the median",
                 x=0.01, ha="left", fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout(); plt.show()
    sens = pd.concat({PARAM_TITLES.get(k, k): cfg_tbl.groupby(k, sort=False)["Sharpe"].agg(["median", "mean", "min", "max"])
                      for k in varied})
    sens.index.names = ["parameter", "value"]
    display(Markdown("**Sharpe across all configurations sharing each parameter value**"))
    display(sens.style.format("{:.2f}"))

# Equity curves: the 5 best configurations, your original rules and the benchmark
fig, ax = plt.subplots(figsize=(11, 4.4))
shown = list(by_sharpe.index[:5])
for k, lab in enumerate(shown):
    ax.plot(GRID[lab]["equity"] / INITIAL_CAPITAL, lw=1.4, color=CATEGORICAL[k], label=lab)
if default_lab in GRID and default_lab not in shown:
    ax.plot(GRID[default_lab]["equity"] / INITIAL_CAPITAL, lw=1.4, color=CATEGORICAL[6], label=f"{default_lab} (your rules)")
bench_eq = (1 + BENCH_R.reindex(R.index).fillna(0)).cumprod()
ax.plot(bench_eq, lw=1.4, color=INK2, label=f"{BENCHMARK}")
growth_axis(ax)
ax.set_title("Growth of $1: top 5 configurations by in-sample Sharpe")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## 6. Walk-forward (step-forward) optimisation
Each fold scores every configuration over the previous `WF_IS_YEARS` years (in-sample) with the `SCORECARD`: highest total return 1 point, smallest max drawdown 1, highest Sharpe 2, highest Calmar 1, lowest beta 0.5, highest Sortino 0.5 (out of 6). The configuration with the most points trades the next `WF_OOS_YEARS` years (out-of-sample), then the window steps forward. The OOS years are simulated as one continuous account. Positions carry over when the parameters change, as they would in live trading. **Only the OOS curve is an honest estimate of performance.**

In [ ]:
def make_folds(first, last):
    folds, first = [], pd.Timestamp(first)
    oos = first + pd.DateOffset(years=WF_IS_YEARS)
    while oos + pd.Timedelta(days=60) <= last:           # a final stub shorter than ~2 months joins the previous fold
        oos_end = min(oos + pd.DateOffset(years=WF_OOS_YEARS) - pd.Timedelta(days=1), last)
        is_start = first if WF_ANCHORED else oos - pd.DateOffset(years=WF_IS_YEARS)
        folds.append((is_start, oos - pd.Timedelta(days=1), oos, oos_end))
        oos += pd.DateOffset(years=WF_OOS_YEARS)
    if folds:
        folds[-1] = folds[-1][:3] + (last,)
    return folds


FOLDS = make_folds(R.index[0], R.index[-1])
schedule, fold_rows = [], []
for is0, is1, oos0, oos1 in FOLDS:
    best, pts, m, won = pick_config(R.loc[is0:is1], BENCH_R)
    schedule.append((oos0, CONFIGS[LABELS.index(best)]))
    fold_rows.append(dict(IS=f"{is0:%Y-%m}..{is1:%Y-%m}", OOS=f"{oos0:%Y-%m}..{oos1:%Y-%m}", chosen=best,
                          **{"IS points": pts[best], "won": ", ".join(won[best]) or "-", "IS Sharpe": m.loc[best, "Sharpe"]},
                          oos0=oos0, oos1=oos1, is0=is0, is1=is1))

WF = run_backtest(FEAT, schedule, start=FOLDS[0][2])
OOS_R = WF["returns"]
B_OOS = BENCH_R.reindex(OOS_R.index).fillna(0)

for row in fold_rows:
    r_oos = OOS_R.loc[row["oos0"]:row["oos1"]]
    b = B_OOS.loc[row["oos0"]:row["oos1"]]
    r_is = R.loc[row["is0"]:row["is1"], row["chosen"]]
    row.update({"IS CAGR": (1 + r_is).prod() ** (TD / max(len(r_is), 1)) - 1,
                "OOS return": (1 + r_oos).prod() - 1, f"{BENCHMARK} return": (1 + b).prod() - 1,
                "OOS Sharpe": perf_stats(r_oos).get("Sharpe", np.nan),
                "OOS max DD": perf_stats(r_oos).get("Max drawdown", np.nan),
                "OOS trades": int(((WF["trades"]["exit_date"] >= row["oos0"]) & (WF["trades"]["exit_date"] <= row["oos1"])).sum())
                if len(WF["trades"]) else 0})
folds_df = pd.DataFrame(fold_rows).drop(columns=["oos0", "oos1", "is0", "is1"])
display(folds_df.style.format({"IS points": "{:.1f}", "IS Sharpe": "{:.2f}", "IS CAGR": "{:.1%}", "OOS return": "{:.1%}",
                               f"{BENCHMARK} return": "{:.1%}", "OOS Sharpe": "{:.2f}", "OOS max DD": "{:.1%}"})
        .hide(axis="index"))

# annualised IS vs OOS return per fold: walk-forward efficiency
yrs_oos = pd.Series([len(OOS_R.loc[r["oos0"]:r["oos1"]]) / TD for r in fold_rows])
oos_ann = pd.Series([(1 + r["OOS return"]) ** (1 / max(y, 1e-9)) - 1 for r, y in zip(fold_rows, yrs_oos)])
is_ann = pd.Series([r["IS CAGR"] for r in fold_rows])
WFE = oos_ann.mean() / is_ann.mean() if is_ann.mean() > 0 else np.nan
stability = folds_df["chosen"].value_counts()
display(Markdown(f"**Walk-forward efficiency** (mean OOS annualised return ÷ mean IS annualised return): **{WFE:.2f}** "
                 "(≥ 0.5 is generally considered robust; near 0 or negative means the IS edge did not carry over).  \n"
                 f"**Parameter stability:** most-chosen configuration `{stability.index[0]}` in "
                 f"{stability.iloc[0]}/{len(folds_df)} folds."))

In [ ]:
# Which value of each parameter was traded in each out-of-sample year
if varied:
    heights = [len(PARAM_GRID[k]) for k in varied]
    fig, axes = plt.subplots(len(varied), 1, figsize=(11, 0.42 * sum(heights) + 0.75 * len(varied)), sharex=True,
                             squeeze=False, gridspec_kw={"height_ratios": heights})
    for ax, key in zip(axes[:, 0], varied):
        vals = PARAM_GRID[key]
        for (d0, prm), row in zip(schedule, fold_rows):
            ax.barh(vals.index(prm[key]), (row["oos1"] - row["oos0"]).days, left=row["oos0"], height=0.55,
                    color=C_STRAT, edgecolor=SURFACE, lw=2)
        ax.set_yticks(range(len(vals)), [fmt_value(key, v) for v in vals])
        ax.set_ylim(-0.6, len(vals) - 0.4)
        ax.invert_yaxis()
        ax.set_title(PARAM_TITLES.get(key, key), fontsize=10)
    axes[-1, 0].xaxis.set_major_locator(mdates.YearLocator(2))
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle("Parameters traded in each out-of-sample year", x=0.01, ha="left", fontsize=12, fontweight="bold",
                 color=INK)
    plt.tight_layout(); plt.show()

# each configuration's return in each OOS window (how consistent is the ranking?)
cons_labels = list(dict.fromkeys([r["chosen"] for r in fold_rows] + ([default_lab] if default_lab in LABELS else [])))
cons = pd.DataFrame({f"{r['oos0']:%Y}": [(1 + R.loc[r["oos0"]:r["oos1"], lab]).prod() - 1 for lab in cons_labels]
                     for r in fold_rows}, index=cons_labels)
fig, ax = plt.subplots(figsize=(11, 0.45 * len(cons_labels) + 1.4))
lim = np.nanmax(np.abs(cons.values)) or 1e-6
ax.imshow(cons.values, cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
ax.set_xticks(range(cons.shape[1]), cons.columns); ax.set_yticks(range(cons.shape[0]), cons.index)
ax.grid(False)
for (a, b), v in np.ndenumerate(cons.values):
    ax.text(b, a, f"{v:.0%}", ha="center", va="center", fontsize=8, color=INK)
for b, r in enumerate(fold_rows):
    a = cons_labels.index(r["chosen"])
    ax.add_patch(plt.Rectangle((b - 0.5, a - 0.5), 1, 1, fill=False, ec=INK, lw=1.5))
ax.set_title("Return per out-of-sample window of every configuration ever picked (outlined = traded that year)")
plt.tight_layout(); plt.show()

### Minimum F-Score: buy at 7+ vs only 8+
Two separate walk-forwards, each allowed to pick only from its own half of the grid: configurations that buy at F-Score ≥ 7 (7, 8 and 9) vs configurations that buy only at F-Score ≥ 8 (8 and 9). Everything else is identical, including the yearly scorecard selection and the exit when the score drops below 7, so this is an out-of-sample, like-for-like answer to "does requiring 8+ help?".

In [ ]:
def walk_forward_subset(keep):
    """Walk-forward restricted to configurations where keep(params) is True (reuses the full-sample grid runs)."""
    labs = [l for l, p in zip(LABELS, CONFIGS) if keep(p)]
    sched = [(oos0, CONFIGS[LABELS.index(pick_config(R.loc[is0:is1, labs], BENCH_R)[0])]) for is0, is1, oos0, _ in FOLDS]
    return run_backtest(FEAT, sched, start=FOLDS[0][2])


if len(PARAM_GRID.get("f_min", [])) > 1:
    FS_RUNS = {f"buy at F-Score >= {v}": walk_forward_subset(lambda p, v=v: p["f_min"] == v) for v in PARAM_GRID["f_min"]}
    fs_tbl = pd.DataFrame({k: {**perf_stats(r["returns"], B_OOS), **trade_stats(r["trades"])} for k, r in FS_RUNS.items()})
    display(Markdown(f"**Walk-forward out-of-sample, {OOS_R.index[0]:%Y-%m-%d} to {OOS_R.index[-1]:%Y-%m-%d}**"))
    display(fmt_table(fs_tbl.reindex(["Total return", "CAGR", "Sharpe", "Sortino", "Max drawdown", "Calmar", "Beta",
                                      "Trades", "Win rate", "Avg trade", "Worst trade", "Avg holding (days)"])))
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for k, (name, res) in enumerate(FS_RUNS.items()):
        eq = (1 + res["returns"]).cumprod()
        axes[0].plot(eq, color=CATEGORICAL[k], label=name)
        axes[1].plot(eq / eq.cummax() - 1, color=CATEGORICAL[k], lw=1.4, label=name)
    axes[0].plot((1 + B_OOS).cumprod(), color=INK2, lw=1.4, label=BENCHMARK)
    growth_axis(axes[0])
    axes[0].set_title("Growth of $1 (walk-forward OOS): minimum F-Score to buy")
    axes[0].legend(loc="upper left")
    pct_axis(axes[1])
    axes[1].set_title("Drawdown")
    axes[1].xaxis.set_major_locator(mdates.YearLocator(2)); axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout(); plt.show()

## 7. Out-of-sample tearsheet (walk-forward result)

In [ ]:
oos_stats = run_summary(WF, B_OOS)
bench_stats = perf_stats(B_OOS)
fixed_res = GRID[default_lab] if default_lab in GRID else run_backtest(FEAT, default_params())
fixed = fixed_res["returns"].reindex(OOS_R.index)
tear = pd.DataFrame({"Walk-forward (OOS)": oos_stats,
                     f"Fixed {default_lab} (same period)": perf_stats(fixed, B_OOS),
                     f"{BENCHMARK} buy & hold": bench_stats})
rows = ["Total return", "CAGR", "Volatility (ann.)", "Sharpe", "Sortino", "Max drawdown", "Calmar",
        "Longest drawdown (days)", "Ulcer index", "Skew", "Excess kurtosis", "VaR 95% (daily)", "CVaR 95% (daily)",
        "Best month", "Worst month", "% positive months", "Beta", "Alpha (ann.)", "Correlation", "Tracking error",
        "Information ratio", "Avg positions", "Avg exposure", "Turnover (x/yr)", "Alpaca fees paid ($)",
        "Slippage paid ($)"]
display(Markdown(f"**Out-of-sample period {OOS_R.index[0]:%Y-%m-%d} to {OOS_R.index[-1]:%Y-%m-%d} "
                 f"({len(OOS_R) / TD:.1f} years), Alpaca fees + {SLIPPAGE_BPS} bp slippage on open orders**"))
display(fmt_table(tear.reindex(rows)))

In [ ]:
eq_s = (1 + OOS_R).cumprod()
eq_b = (1 + B_OOS).cumprod()
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
ax = axes[0]
ax.plot(eq_s, color=C_STRAT, label="Strategy (walk-forward OOS)")
ax.plot(eq_b, color=C_BENCH, label=f"{BENCHMARK} buy & hold")
growth_axis(ax)
for s, c in ((eq_s, C_STRAT), (eq_b, C_BENCH)):
    ax.plot(s.index[-1], s.iloc[-1], "o", ms=6, color=c, mec=SURFACE, mew=2)
    ax.annotate(f"{s.iloc[-1]:.2f}x", (s.index[-1], s.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", color=INK2, fontsize=9)
ax.set_title("Growth of $1 (log scale)")
ax.legend(loc="upper left")
ax = axes[1]
for s, c, lab in ((eq_s, C_STRAT, "Strategy"), (eq_b, C_BENCH, BENCHMARK)):
    dd = s / s.cummax() - 1
    ax.plot(dd, color=c, lw=1.4, label=lab)
    if c == C_STRAT:
        ax.fill_between(dd.index, dd, 0, color=c, alpha=0.10, lw=0)
pct_axis(ax)
ax.set_title("Drawdown")
ax.legend(loc="lower left")
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()

In [ ]:
# Monthly returns heatmap + annual returns vs benchmark
m = (1 + OOS_R).groupby([OOS_R.index.year, OOS_R.index.month]).prod() - 1
mt = m.unstack().reindex(columns=range(1, 13))
mt.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
yr = (1 + OOS_R).groupby(OOS_R.index.year).prod() - 1
yb = (1 + B_OOS).groupby(B_OOS.index.year).prod() - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 0.38 * len(mt) + 1.6), gridspec_kw={"width_ratios": [2.4, 1]})
ax = axes[0]
lim = np.nanmax(np.abs(mt.values)) or 1e-6
ax.imshow(mt.values, cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
ax.set_xticks(range(12), mt.columns); ax.set_yticks(range(len(mt)), mt.index)
ax.grid(False)
for (a, b), v in np.ndenumerate(mt.values):
    if np.isfinite(v):
        ax.text(b, a, f"{v:.1%}", ha="center", va="center", fontsize=7.5, color=INK)
ax.set_title("Monthly returns (OOS)")
ax = axes[1]
y = np.arange(len(yr))
ax.barh(y - 0.2, yr.values, height=0.38, color=C_STRAT, label="Strategy")
ax.barh(y + 0.2, yb.reindex(yr.index).values, height=0.38, color=C_BENCH, label=BENCHMARK)
ax.set_yticks(y, yr.index); ax.invert_yaxis()
ax.axvline(0, color=AXIS, lw=0.8)
pct_axis(ax, "x")
ax.set_title("Calendar-year return")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Rolling risk-adjusted return, exposure and sector mix
roll = OOS_R.rolling(TD)
roll_sh = roll.mean() / roll.std() * np.sqrt(TD)
roll_b = B_OOS.rolling(TD).mean() / B_OOS.rolling(TD).std() * np.sqrt(TD)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax = axes[0]
ax.plot(roll_sh, color=C_STRAT, label="Strategy"); ax.plot(roll_b, color=C_BENCH, label=BENCHMARK)
ax.axhline(0, color=AXIS, lw=0.8)
ax.set_title("Rolling 12-month Sharpe ratio"); ax.legend(loc="upper left")
ax = axes[1]
ax.plot(WF["n_pos"].rolling(5).mean(), color=C_STRAT, lw=1.4)
ax.set_ylim(0, max(p["max_positions"] for _, p in schedule) + 0.5)
ax.set_title("Positions held (5-day average)")
ax = axes[2]
names = FEAT["sector_names"]
arr = np.zeros((len(WF["n_pos"]), len(names)))
for d, h in enumerate(WF["holdings"]):
    for j in h:
        arr[d, FEAT["sector_code"][j]] += 1
sec_w = pd.DataFrame(arr, index=WF["n_pos"].index, columns=names)
sec_w = sec_w.groupby(sec_w.index.to_period("M")).mean()
sec_w.index = sec_w.index.to_timestamp()
sec_w = sec_w.loc[:, sec_w.sum() > 0]
top = sec_w.sum().sort_values(ascending=False)
keep = list(top.index[:7])
if len(top) > 7:
    sec_w["Other"] = sec_w.drop(columns=keep).sum(axis=1)
    keep.append("Other")
colors = CATEGORICAL[:7] + ["#b9b8b2"]
ax.stackplot(sec_w.index, sec_w[keep].T.values, labels=keep, colors=colors[:len(keep)], edgecolor=SURFACE, lw=0.6)
ax.set_title("Holdings by sector (monthly average count)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.show()

### Trade analysis (OOS)

In [ ]:
TR = WF["trades"]
if len(TR):
    ts = pd.Series(trade_stats(TR))
    display(fmt_table(ts.to_frame("Walk-forward OOS")))
    reasons = TR.groupby("reason").agg(trades=("ret", "size"), win_rate=("ret", lambda x: (x > 0).mean()),
                                       avg_return=("ret", "mean"), avg_days=("days", "mean"))
    display(reasons.style.format({"win_rate": "{:.1%}", "avg_return": "{:.2%}", "avg_days": "{:.0f}"}))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    ax = axes[0]
    bins = np.linspace(TR["ret"].quantile(0.005), TR["ret"].quantile(0.995), 40)
    ax.hist(TR["ret"].clip(bins[0], bins[-1]), bins=bins, color=C_STRAT, edgecolor=SURFACE, lw=1)
    ax.axvline(0, color=AXIS, lw=0.8)
    ax.axvline(TR["ret"].mean(), color=INK2, lw=1.2)
    ax.annotate(f"mean {TR['ret'].mean():.2%}", (TR["ret"].mean(), ax.get_ylim()[1] * 0.92), xytext=(4, 0),
                textcoords="offset points", color=INK2, fontsize=9)
    pct_axis(ax, "x")
    ax.set_title("Trade return distribution (net of costs)")
    ax.set_ylabel("trades")
    ax = axes[1]
    ax.hist(TR["days"], bins=40, color=C_STRAT, edgecolor=SURFACE, lw=0.5)
    ax.set_title("Holding period (calendar days)")
    ax.set_ylabel("trades")
    plt.tight_layout(); plt.show()

    show_cols = ["ticker", "sector", "entry_date", "exit_date", "entry_px", "exit_px", "ret", "weight", "days", "reason",
                 "f_entry", "mcap_bn", "config"]
    fmt = {"entry_px": "{:.2f}", "exit_px": "{:.2f}", "ret": "{:.2%}", "weight": "{:.1%}", "f_entry": "{:.0f}", "mcap_bn": "{:,.0f}",
           "entry_date": "{:%Y-%m-%d}", "exit_date": "{:%Y-%m-%d}"}
    def worst_day(r):
        """Largest one-day price drop while held (close to close, entry day from the fill price)."""
        j, i, k = list(FEAT["tickers"]).index(r.ticker), FEAT["dates"].get_loc(r.entry_date), FEAT["dates"].get_loc(r.exit_date)
        px = np.r_[r.entry_px, FEAT["close"][i:k, j], r.exit_px]
        return np.nanmin(px[1:] / px[:-1] - 1)

    extremes = pd.concat([TR.nlargest(10, "ret"), TR.nsmallest(10, "ret")]).copy()
    extremes["worst_day"] = [worst_day(r) for r in extremes.itertuples()]
    display(Markdown("**Best and worst 10 trades.** `worst_day` is the largest one-day drop while held. A drop of "
                     "around -50% (2-for-1) or -67% (3-for-1) with no news usually means Yahoo missed a stock split, so "
                     "check the chart before trusting that trade."))
    display(extremes[show_cols[:7] + ["worst_day"] + show_cols[7:]].style.format({**fmt, "worst_day": "{:.1%}"})
            .hide(axis="index"))
    display(Markdown("**Most traded stocks**"))
    display(TR.groupby("ticker").agg(trades=("ret", "size"), avg_return=("ret", "mean"), total_pnl=("pnl", "sum"))
              .sort_values("trades", ascending=False).head(15)
              .style.format({"avg_return": "{:.2%}", "total_pnl": "{:,.0f}"}))
else:
    print("No trades in the out-of-sample period.")

## 8. Overfitting diagnostics

* **Probabilistic Sharpe Ratio (PSR)**: probability that the true Sharpe is above 0, given the track length, skew and fat tails (Bailey & López de Prado, 2012).
* **Deflated Sharpe Ratio (DSR)**: PSR measured against the Sharpe you would expect from the *best of N* skill-less configurations. It corrects for having tried N settings, 180 by default (Bailey & López de Prado, 2014). The more settings tried, the higher the bar.
* **Probability of Backtest Overfitting (PBO)** via combinatorially-symmetric cross-validation (CSCV), using Sharpe as the selection rule. The sample is split into 16 blocks, and every half/half split is tried. PBO is the share of splits where the in-sample winner ranks in the bottom half out-of-sample (Bailey, Borwein, López de Prado & Zhu, 2015). Below about 0.2 is comfortable; 0.5 means the selection is no better than a coin flip.
* **Block bootstrap**: 21-day blocks of OOS daily returns are resampled to get confidence intervals for Sharpe, CAGR and max drawdown.
* **Cost sensitivity**: the OOS schedule is re-run with more slippage on the market-on-open orders, on top of Alpaca's fees.
* **Risk-control ablation**: the full walk-forward is re-run with the stop loss and market filter switched off, to check they help out-of-sample.

In [ ]:
EMC = 0.5772156649


def psr(r, sr_ref=0.0):
    r = r.dropna()
    sr = r.mean() / r.std()
    g3, g4 = stats.skew(r), stats.kurtosis(r, fisher=False)
    return stats.norm.cdf((sr - sr_ref) * np.sqrt(len(r) - 1) / np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2))


def deflated_sharpe(R):
    """DSR of the best column of R (daily returns of all trials)."""
    sr = R.mean() / R.std()
    n = R.shape[1]
    sr0 = np.sqrt(sr.var(ddof=1)) * ((1 - EMC) * stats.norm.ppf(1 - 1 / n) + EMC * stats.norm.ppf(1 - 1 / (n * np.e)))
    best = sr.idxmax()
    return best, psr(R[best], sr0), sr0 * np.sqrt(TD)


def pbo_cscv(R, S=CSCV_BLOCKS):
    X = R.dropna().to_numpy()
    T, N = X.shape
    blocks = np.array_split(np.arange(T), S)
    s1 = np.array([X[b].sum(0) for b in blocks]); s2 = np.array([(X[b] ** 2).sum(0) for b in blocks])
    cnt = np.array([len(b) for b in blocks], float)

    def sharpe(mask):
        n = cnt[mask].sum(); mu = s1[mask].sum(0) / n
        return mu / np.sqrt(np.maximum(s2[mask].sum(0) / n - mu ** 2, 1e-18))

    logits = []
    for comb in itertools.combinations(range(S), S // 2):
        m = np.zeros(S, bool); m[list(comb)] = True
        best = np.argmax(sharpe(m))
        w = stats.rankdata(sharpe(~m))[best] / (N + 1)
        logits.append(np.log(w / (1 - w)))
    logits = np.array(logits)
    return (logits <= 0).mean(), logits


def block_bootstrap(r, n=N_BOOTSTRAP, block=21, seed=42):
    x = r.dropna().to_numpy()
    T = len(x)
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, T, size=(n, int(np.ceil(T / block))))
    idx = ((starts[:, :, None] + np.arange(block)) % T).reshape(n, -1)[:, :T]
    sims = x[idx]
    eq = np.cumprod(1 + sims, axis=1)
    return pd.DataFrame({"Sharpe": sims.mean(1) / sims.std(1) * np.sqrt(TD),
                         "CAGR": eq[:, -1] ** (TD / T) - 1,
                         "Max drawdown": (eq / np.maximum.accumulate(eq, axis=1) - 1).min(1)})


R_full = R.dropna()
best_cfg, dsr, sr0_ann = deflated_sharpe(R_full)
pbo, logits = pbo_cscv(R_full)
boot = block_bootstrap(OOS_R)
ci = boot.quantile([0.05, 0.5, 0.95]).T
ci.columns = ["5%", "median", "95%"]

diag = pd.DataFrame({"value": {
    "PSR (OOS walk-forward, SR > 0)": psr(OOS_R),
    f"Best in-sample config": best_cfg,
    "Best in-sample Sharpe (full sample)": perf_stats(R_full[best_cfg])["Sharpe"],
    f"Expected max Sharpe of {R_full.shape[1]} skill-less trials": sr0_ann,
    "Deflated Sharpe Ratio of best config": dsr,
    "PBO (CSCV)": pbo,
    "Walk-forward efficiency": WFE,
}})
display(diag.style.format(lambda v: f"{v:.3f}" if isinstance(v, (float, np.floating)) else v))
display(Markdown("**Block-bootstrap 90% confidence intervals, OOS walk-forward**"))
display(ci.style.format("{:.2f}", subset=(["Sharpe"], ci.columns))
             .format("{:.1%}", subset=(["CAGR", "Max drawdown"], ci.columns)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
ax = axes[0]
ax.hist(logits, bins=25, color=C_STRAT, edgecolor=SURFACE, lw=1)
ax.axvline(0, color=INK2, lw=1.2)
ax.set_title(f"CSCV logits (PBO = {pbo:.2f})")
ax.set_xlabel("logit of OOS rank of the IS winner")
ax = axes[1]
ax.hist(boot["Sharpe"], bins=40, color=C_STRAT, edgecolor=SURFACE, lw=1)
ax.axvline(0, color=AXIS, lw=0.8)
ax.axvline(perf_stats(OOS_R)["Sharpe"], color=INK2, lw=1.2)
ax.set_title("Bootstrap Sharpe (OOS)")
ax = axes[2]
ax.hist(boot["Max drawdown"], bins=40, color=C_STRAT, edgecolor=SURFACE, lw=1)
pct_axis(ax, "x")
ax.set_title("Bootstrap max drawdown (OOS)")
plt.tight_layout(); plt.show()

In [ ]:
# Cost sensitivity: same walk-forward schedule, Alpaca fees always on, different slippage on open orders
cost_rows = {}
for bps in (0, 1, 2, 5, 10):
    res = run_backtest(FEAT, schedule, start=FOLDS[0][2], slippage_bps=bps)
    s = perf_stats(res["returns"], B_OOS)
    cost_rows[f"{bps} bps slippage"] = {**{k: s[k] for k in ("CAGR", "Sharpe", "Max drawdown", "Alpha (ann.)")},
                                        "Alpaca fees paid ($)": res["fees"], "Slippage paid ($)": res["slippage"]}
display(Markdown("**Cost sensitivity (walk-forward OOS): Alpaca fees + different slippage on market-on-open orders**"))
display(fmt_table(pd.DataFrame(cost_rows)))

# Execution-rule sensitivity: take-profit filled only on a close above SMA200 (next open)
alt = run_backtest(FEAT, schedule, start=FOLDS[0][2], tp_mode="close" if TP_MODE == "limit" else "limit")
display(Markdown(f"**Take-profit rule sensitivity**: `{TP_MODE}` (base) vs "
                 f"`{'close' if TP_MODE == 'limit' else 'limit'}`"))
display(fmt_table(pd.DataFrame({TP_MODE: perf_stats(OOS_R, B_OOS),
                                ("close" if TP_MODE == "limit" else "limit"): perf_stats(alt["returns"], B_OOS)})
                  .loc[["CAGR", "Sharpe", "Max drawdown", "Alpha (ann.)"]]))

### Do the risk controls help? (walk-forward ablation)
The market filter, time limit and stop loss are fixed rules, chosen before looking at results. Each row below re-runs the **entire** walk-forward with **one** of them flipped relative to the current settings: all configurations, the yearly parameter selection, and the out-of-sample account. The comparison is therefore out-of-sample, like the headline result. A control earns its place if it cuts the tail (worst trade, worst month, max drawdown) without giving up much Sharpe.

Choosing among these variants after seeing the table is itself a (small) in-sample choice. Prefer the variant you would have picked on reasoning alone, unless the difference is large.

In [ ]:
def walk_forward_variant(**kw):
    """Full walk-forward (grid -> yearly selection -> OOS account) under different risk-control settings."""
    Rv = pd.DataFrame({lab: run_backtest(FEAT, p, **kw)["returns"] for lab, p in zip(LABELS, CONFIGS)})
    sched = [(oos0, CONFIGS[LABELS.index(pick_config(Rv.loc[is0:is1], BENCH_R)[0])]) for is0, is1, oos0, _ in FOLDS]
    return run_backtest(FEAT, sched, start=FOLDS[0][2], **kw)


def risk_summary(res):
    s = perf_stats(res["returns"], B_OOS)
    tr = res["trades"]
    return {"CAGR": s["CAGR"], "Sharpe": s["Sharpe"], "Sortino": s["Sortino"], "Max drawdown": s["Max drawdown"],
            "Calmar": s["Calmar"], "Worst month": s["Worst month"],
            "Worst trade": tr["ret"].min() if len(tr) else np.nan, "Avg trade": tr["ret"].mean() if len(tr) else np.nan,
            "Trades": len(tr), "Stopped out": (tr["reason"] == "stop loss").mean() if len(tr) else np.nan,
            "Hit time limit": (tr["reason"] == "time limit").mean() if len(tr) else np.nan,
            "Longest trade (days)": tr["days"].max() if len(tr) else np.nan,
            "Avg exposure": res["exposure"].mean(),
            "Drawdowns worse than -20%": drawdown_episodes(res["returns"], -0.20),
            "Median mcap at entry ($bn)": tr["mcap_bn"].median() if len(tr) else np.nan}


def drawdown_episodes(r, level):
    """Number of separate times the equity curve fell more than `level` below its previous peak."""
    eq = (1 + r).cumprod()
    under = (eq / eq.cummax() - 1) < level
    return int((under & ~under.shift(fill_value=False)).sum())


if RUN_RISK_ABLATION:
    current = dict(stop_loss=STOP_LOSS, market_filter=MARKET_FILTER, max_hold=MAX_HOLD_DAYS)
    flip = lambda **kw: {**current, **kw}
    variants = {"current rules": current,
                ("without" if MARKET_FILTER else "with") + " market filter":
                    flip(market_filter=None if MARKET_FILTER else "no_new_entries"),
                ("without" if MAX_HOLD_DAYS else "with") + " 1-year time limit": flip(max_hold=None if MAX_HOLD_DAYS else 252),
                ("without" if STOP_LOSS else "with") + " 10% stop": flip(stop_loss=None if STOP_LOSS else 0.10)}
    if MARKET_FILTER != "liquidate":                    # drawdown protection: sell everything when SPY breaks its SMA200
        variants["market filter sells everything"] = flip(market_filter="liquidate")
    t0 = time.time()
    ABL = {name: WF if kw == current else walk_forward_variant(**kw) for name, kw in variants.items()}
    print(f"Ablation: {sum(r is not WF for r in ABL.values())} extra walk-forward runs in {time.time() - t0:.0f}s")
    PCT_ROWS |= {"Stopped out", "Hit time limit"}
    display(Markdown(f"**Walk-forward out-of-sample, {OOS_R.index[0]:%Y-%m-%d} to {OOS_R.index[-1]:%Y-%m-%d}** "
                     f"(current settings: MARKET_FILTER={MARKET_FILTER!r}, MAX_HOLD_DAYS={MAX_HOLD_DAYS}, "
                     f"STOP_LOSS={STOP_LOSS})"))
    display(fmt_table(pd.DataFrame({k: risk_summary(v) for k, v in ABL.items()})))

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for k, (name, res) in enumerate(ABL.items()):
        eq = (1 + res["returns"]).cumprod()
        axes[0].plot(eq, color=CATEGORICAL[k], lw=1.6, label=name)
        axes[1].plot(eq / eq.cummax() - 1, color=CATEGORICAL[k], lw=1.2, label=name)
    growth_axis(axes[0])
    axes[0].set_title("Growth of $1 (walk-forward OOS): current rules vs one change at a time")
    axes[0].legend(loc="upper left")
    pct_axis(axes[1])
    axes[1].set_title("Drawdown")
    off = ~ABL["current rules"]["risk_on"]
    for ax in axes:                                   # shade the days the market filter was risk-off
        ax.fill_between(off.index, 0, 1, where=off.values, transform=ax.get_xaxis_transform(), color=MUTED,
                        alpha=0.12, lw=0)
    axes[1].xaxis.set_major_locator(mdates.YearLocator(2)); axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout(); plt.show()
    print(f"Grey bands: {BENCHMARK} below its {MARKET_FILTER_SMA}-day SMA ({off.mean():.0%} of days).")

In [ ]:
# Stop level sensitivity on the walk-forward schedule above. For information only: picking the best row would be
# fitting the stop to history. A good stop rule should look similar at neighbouring levels.
stop_rows = {}
for lvl in (None, 0.05, 0.10, 0.15, 0.20, 0.25):
    res = run_backtest(FEAT, schedule, start=FOLDS[0][2], stop_loss=lvl)
    stop_rows["no stop" if lvl is None else f"{lvl:.0%}"] = risk_summary(res)
display(Markdown("**Stop-loss level sensitivity (walk-forward schedule fixed, information only)**"))
display(fmt_table(pd.DataFrame(stop_rows).loc[["CAGR", "Sharpe", "Max drawdown", "Worst month", "Worst trade",
                                               "Avg trade", "Stopped out"]]))

### Ranking and entry-trigger variants
Each row re-runs the **entire** walk-forward (all configurations, yearly scorecard selection, out-of-sample account) with one rule changed:
* **Rank by expected upside**: candidates ordered by how far they are below their SMA200 (the profit target), largest gap first, instead of by market cap.
* **Rank by F-Score**: highest F-Score first, market cap breaking ties.
* **Buy only after an F-Score upgrade**: a stock qualifies only if its latest filing lifted the F-Score from below 7 to the minimum or above. It stays eligible until the next filing.

The universe is the point-in-time S&P 500 in every row, so every pick is a large cap (the median market cap at entry is shown to confirm it).

In [ ]:
if RUN_VARIANTS:
    base_kw = dict(rank_by=RANK_BY, entry_trigger=ENTRY_TRIGGER)
    variant_kw = {f"current rules (rank by {RANK_BY.replace('_', ' ')}, entry on F-Score level)": base_kw}
    for rb in ("market_cap", "upside", "fscore"):
        if rb != RANK_BY:
            variant_kw[f"rank by {rb.replace('_', ' ')}"] = {**base_kw, "rank_by": rb}
    other = "upgrade" if ENTRY_TRIGGER == "level" else "level"
    variant_kw[f"entry on F-Score {other}"] = {**base_kw, "entry_trigger": other}
    t0 = time.time()
    VAR = {name: WF if kw == base_kw else walk_forward_variant(**kw) for name, kw in variant_kw.items()}
    print(f"Variants: {sum(r is not WF for r in VAR.values())} extra walk-forward runs in {time.time() - t0:.0f}s")
    display(Markdown(f"**Walk-forward out-of-sample, {OOS_R.index[0]:%Y-%m-%d} to {OOS_R.index[-1]:%Y-%m-%d}**"))
    display(fmt_table(pd.DataFrame({k: risk_summary(v) for k, v in VAR.items()})))

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for k, (name, res) in enumerate(VAR.items()):
        eq = (1 + res["returns"]).cumprod()
        axes[0].plot(eq, color=CATEGORICAL[k], lw=1.6, label=name)
        axes[1].plot(eq / eq.cummax() - 1, color=CATEGORICAL[k], lw=1.2, label=name)
    growth_axis(axes[0])
    axes[0].set_title("Growth of $1 (walk-forward OOS): ranking and entry-trigger variants")
    axes[0].legend(loc="upper left", fontsize=8)
    pct_axis(axes[1])
    axes[1].set_title("Drawdown")
    axes[1].xaxis.set_major_locator(mdates.YearLocator(2)); axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout(); plt.show()

## 9. Lookahead-bias audit
The whole pipeline is rebuilt using only information available on an audit date *D*: prices up to *D*, SEC facts **filed** up to *D*, and index membership up to *D*. The backtest is then re-run. If any step used future data, the equity curve up to *D* would differ from the full-data run. The two must match to the cent.

*Split events are kept in both runs. They only restate share counts in the same units as Yahoo's split-adjusted prices, and the resulting market cap depends only on splits between the share-count date and day t.*

In [ ]:
if RUN_LOOKAHEAD_AUDIT:
    AUDIT_DATE = FEAT["dates"][int(len(FEAT["dates"]) * 0.6)]
    print(f"Rebuilding fundamentals and features with data up to {AUDIT_DATE:%Y-%m-%d} ...")
    fund_cut = compute_fundamentals(companies, cutoff=AUDIT_DATE, verbose=False)
    FEAT_CUT = build_features(price_frames, fund_cut, membership.loc[:AUDIT_DATE], cutoff=AUDIT_DATE)
    tick_full = list(FEAT["tickers"])
    audit_rows = []
    for name, sched, start in (("default config", default_params(), None),
                               ("walk-forward schedule", schedule, FOLDS[0][2])):
        full = run_backtest(FEAT, sched, start=start, end=AUDIT_DATE)
        cut = run_backtest(FEAT_CUT, sched, start=start, end=AUDIT_DATE)
        diff = (full["equity"] - cut["equity"].reindex(full["equity"].index)).abs().max()
        same_trades = (len(full["trades"]) == len(cut["trades"]) and
                       (len(full["trades"]) == 0 or
                        (full["trades"][["ticker", "entry_date", "exit_date"]].reset_index(drop=True)
                         .equals(cut["trades"][["ticker", "entry_date", "exit_date"]].reset_index(drop=True)))))
        first = ""
        if not same_trades:                              # first trade that differs, to make a failure traceable
            cols = ["ticker", "entry_date", "exit_date"]
            a_ = full["trades"][cols].astype(str).agg(" ".join, axis=1).tolist() if len(full["trades"]) else []
            b_ = cut["trades"][cols].astype(str).agg(" ".join, axis=1).tolist() if len(cut["trades"]) else []
            n_ = next((x for x in range(min(len(a_), len(b_))) if a_[x] != b_[x]), min(len(a_), len(b_)))
            first = f"full: {a_[n_] if n_ < len(a_) else '-'} | cut: {b_[n_] if n_ < len(b_) else '-'}"
        audit_rows.append({"run": name, "days compared": len(full["equity"]), "max |equity diff|": diff,
                           "trades (full / cut)": f"{len(full['trades'])} / {len(cut['trades'])}",
                           "identical trades": same_trades, "PASS": bool(diff < 1e-6 and same_trades),
                           "first difference": first})
    # feature-level check on the stocks present in both builds
    common = [t for t in FEAT_CUT["tickers"] if t in tick_full]
    jf = [tick_full.index(t) for t in common]
    jc = [list(FEAT_CUT["tickers"]).index(t) for t in common]
    n_days = len(FEAT_CUT["dates"])
    for key in ("fscore", "fscore_prev", "mcap", "sma200"):
        a, b = FEAT[key][:n_days, jf], FEAT_CUT[key][:, jc]
        a_, b_ = np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1)
        same = np.allclose(a_, b_, rtol=1e-9, atol=1e-6)
        first = ""
        if not same:
            d_, j_ = np.argwhere(~np.isclose(a_, b_, rtol=1e-9, atol=1e-6))[0]
            first = (f"{common[j_]} on {FEAT_CUT['dates'][d_]:%Y-%m-%d}: full {a[d_, j_]:.6g} vs cut {b[d_, j_]:.6g} "
                     f"({int((~np.isclose(a_, b_, rtol=1e-9, atol=1e-6)).any(axis=0).sum())} stocks differ)")
        audit_rows.append({"run": f"feature: {key}", "days compared": n_days, "PASS": bool(same), "first difference": first})
    for key in ("vol_ref", "market_ok"):                 # market-wide series used by sizing and the market filter
        a, b = np.asarray(FEAT[key][:n_days], float), np.asarray(FEAT_CUT[key], float)
        same = np.allclose(np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1), rtol=1e-9, atol=1e-12)
        audit_rows.append({"run": f"feature: {key}", "days compared": n_days, "PASS": bool(same)})
    audit = pd.DataFrame(audit_rows).fillna("")
    display(audit.style.hide(axis="index"))
    print("LOOKAHEAD AUDIT:", "PASS" if audit["PASS"].all() else "FAIL - investigate before trusting results")

## 10. Current signals (live screen)
The configuration chosen by the latest walk-forward fold, applied to the most recent close. These are the stocks the strategy would hold or buy next, assuming the walk-forward account above.

In [ ]:
live = schedule[-1][1]
i = len(FEAT["dates"]) - 1
with np.errstate(invalid="ignore"):
    cand = np.flatnonzero(FEAT["base"][i] & FEAT["above"][live["sma_fast"]][i] & (FEAT["fscore"][i] >= live["f_min"]))
screen = pd.DataFrame({
    "ticker": FEAT["tickers"][cand], "sector": [FEAT["sector_names"][FEAT["sector_code"][j]] for j in cand],
    "mcap_bn": FEAT["mcap"][i, cand] / 1e9, "F-Score": FEAT["fscore"][i, cand], "close": FEAT["close"][i, cand],
    f"SMA{live['sma_fast']}": FEAT["sma"][live["sma_fast"]][i, cand], "SMA200 (target)": FEAT["sma200"][i, cand],
}).sort_values("mcap_bn", ascending=False)
screen["upside to target"] = screen["SMA200 (target)"] / screen["close"] - 1
display(Markdown(f"**As of {FEAT['dates'][i]:%Y-%m-%d}, configuration `{label(live)}`: {len(screen)} candidates**"))
if MARKET_FILTER and not FEAT["market_ok"][i]:
    display(Markdown(f"⚠️ **Market filter is risk-off** ({BENCHMARK} is below its {MARKET_FILTER_SMA}-day SMA): "
                     "the strategy would not buy any of these today."))
display(screen.head(25).style.format({"mcap_bn": "{:,.0f}", "F-Score": "{:.0f}", "close": "{:.2f}",
                                      f"SMA{live['sma_fast']}": "{:.2f}", "SMA200 (target)": "{:.2f}",
                                      "upside to target": "{:.1%}"}).hide(axis="index"))
if len(WF["open"]):
    display(Markdown("**Open positions in the walk-forward account**"))
    display(WF["open"].style.format({"entry_px": "{:.2f}", "last_close": "{:.2f}", "unrealised": "{:.2%}",
                                     "f_now": "{:.0f}", "entry_date": "{:%Y-%m-%d}"}).hide(axis="index"))

## 11. Export

In [ ]:
pd.DataFrame({"strategy_oos": eq_s, BENCHMARK: eq_b}).to_csv(REPORT_DIR / "equity_oos.csv")
WF["trades"].to_csv(REPORT_DIR / "trades_oos.csv", index=False)
tear.to_csv(REPORT_DIR / "metrics_oos.csv")
grid_tbl.to_csv(REPORT_DIR / "grid_full_sample.csv")
folds_df.to_csv(REPORT_DIR / "walk_forward_folds.csv", index=False)
diag.to_csv(REPORT_DIR / "overfitting_diagnostics.csv")
print("Saved to", REPORT_DIR.resolve())
print("For a code-free HTML report:  jupyter nbconvert --to html --no-input SMA_Piotroski_Backtest.ipynb")

## 12. Limitations (read before trusting the numbers)
* **Survivorship bias (partial).** Membership is point-in-time, but Yahoo Finance has no prices for most delisted or acquired companies, so those names drop out of the test. Most of them left the index after declines, so results are likely **somewhat optimistic**. Removing this bias needs paid data (Norgate, CRSP, EOD Historical Data).
* **Sectors are today's GICS classification** (Yahoo's sector for former members). Reclassifications, such as Alphabet and Meta moving to Communication Services in 2018, are applied retroactively. This affects only the per-sector cap.
* **XBRL tagging varies by company.** The tag lists cover most filers; a missing input scores 0 on that test, which makes the screen stricter.
* **Fills.** Opens and limit fills assume enough liquidity (reasonable for S&P 500 names at this size). Costs are Alpaca's 2026 fee schedule plus order-type slippage. Regulatory fee rates were different in earlier years (the SEC rate is reset annually), which changes results only marginally. Take-profit limits are assumed to fill whenever the day's high reaches the limit; in practice a limit that is only just touched may not fill (queue position), a small optimistic bias. Taxes and borrowing are ignored, and idle cash earns 0%.
* **Ticker reuse and renames.** `TICKER_ALIASES` maps known renames; an unmapped rename makes that company look like it left the index.
* Past performance, even out-of-sample, does not guarantee future results. This notebook is research, not investment advice.